# Prueba algoritmos de optimización  

### Imports

Se va a usar gurobipy para esta implementación.

In [ ]:
!pip install gurobipy
#!pip install numpy
#!pip install pandas
#!pip install matplotlib


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Definición de los algoritmos

### Solo variar vector de costos

In [ ]:
def algorithm1(
    a_hat, b_hat, c_hat, H_bounds, M,
    tol=1e-6, time_limit_seconds=120,
    favorable_constraints=None,
    c_signs=None,
    bounds=None,
    var_types=None
):
    """
    Implementación fiel del Algorithm 1 (Solve Problem (4)).

    - Minimización L1 en c
    - Separación por MIN c^T y
    - Cortes: c^T x <= c^T y
    """

    import time
    import numpy as np
    import gurobipy as gp
    from gurobipy import GRB

    t_start = time.perf_counter()

    # Seguridad básica
    n = len(c_hat)
    Y = []
    optimal = False
    iteration = 0

    # Defaults seguros
    if favorable_constraints is None:
        favorable_constraints = []

    if bounds is None or var_types is None:
        raise RuntimeError("bounds y var_types deben ser provistos explícitamente")

    # A, b desde el entorno (como en tu código original)
    A_glob = globals().get("A", None)
    b_glob = globals().get("b", None)
    if A_glob is None or b_glob is None:
        raise RuntimeError("Faltan A o b en el entorno global")

    # ============================
    # BUCLE PRINCIPAL
    # ============================
    while not optimal:
        iteration += 1

        if (time.perf_counter() - t_start) > time_limit_seconds:
            return {"status": "TIME_LIMIT", "sol": None, "iters": iteration}

        # ============================
        # MASTER PROBLEM (Paso 4)
        # ============================
        master = gp.Model("master")
        master.Params.OutputFlag = 0

        # Variables x ∈ D ∩ X
        x = []
        for i in range(n):
            lb, ub = bounds[i]
            if var_types[i] == "B":
                x.append(master.addVar(vtype=GRB.BINARY, name=f"x_{i}"))
            elif var_types[i] == "I":
                x.append(master.addVar(lb=lb, ub=ub, vtype=GRB.INTEGER, name=f"x_{i}"))
            else:
                x.append(master.addVar(lb=lb, ub=ub, name=f"x_{i}"))

        # Variables c y auxiliares L1
        c = master.addVars(n, lb=-GRB.INFINITY, name="c")
        u = master.addVars(n, lb=0, name="u")

        # ||c - c_hat||_1
        for i in range(n):
            master.addConstr(c[i] - c_hat[i] <= u[i])
            master.addConstr(c_hat[i] - c[i] <= u[i])

        master.setObjective(gp.quicksum(u[i] for i in range(n)), GRB.MINIMIZE)

        # a^T x >= b
        for i in range(len(b_glob)):
            master.addConstr(
                gp.quicksum(A_glob[i, j] * x[j] for j in range(n)) >= b_glob[i]
            )

        # x ∈ D (conjunto favorable)
        for idx, fc in enumerate(favorable_constraints):
            expr = gp.quicksum(fc["coeffs"][j] * x[j] for j in range(n))
            if fc["sense"] == ">=":
                master.addConstr(expr >= fc["rhs"])
            elif fc["sense"] == "<=":
                master.addConstr(expr <= fc["rhs"])
            else:
                master.addConstr(expr == fc["rhs"])

        # c ∈ H_c (signos)
        if c_signs is not None:
            for i in range(n):
                if c_signs[i] == ">=":
                    master.addConstr(c[i] >= 0)
                elif c_signs[i] == "<=":
                    master.addConstr(c[i] <= 0)

        # Cortes de optimalidad: ĉᵀ x ≤ ĉᵀ y
        for idx, y_sol in enumerate(Y):
            master.addConstr(
                gp.quicksum(c[i] * x[i] for i in range(n))
                <= gp.quicksum(c[i] * y_sol[i] for i in range(n)),
                name=f"cut_{idx}"
            )

        master.optimize()
        if master.status != GRB.OPTIMAL:
            return {"status": master.status, "sol": None, "iters": iteration}

        c_sol = np.array([c[i].X for i in range(n)])
        x_sol = np.array([x[i].X for i in range(n)])
        obj_master = float(np.dot(c_sol, x_sol))

        # ============================
        # SEPARATION PROBLEM (Paso 5)
        # ============================
        sep = gp.Model("separation")
        sep.Params.OutputFlag = 0

        y = []
        for i in range(n):
            lb, ub = bounds[i]
            if var_types[i] == "B":
                y.append(sep.addVar(vtype=GRB.BINARY))
            elif var_types[i] == "I":
                y.append(sep.addVar(lb=lb, ub=ub, vtype=GRB.INTEGER))
            else:
                y.append(sep.addVar(lb=lb, ub=ub))

        for i in range(len(b_glob)):
            sep.addConstr(
                gp.quicksum(A_glob[i, j] * y[j] for j in range(n)) >= b_glob[i]
            )

        sep.setObjective(
            gp.quicksum(c_sol[i] * y[i] for i in range(n)),
            GRB.MINIMIZE
        )

        sep.optimize()
        if sep.status != GRB.OPTIMAL:
            optimal = True
            break

        y_sol = np.array([y[i].X for i in range(n)])
        obj_sep = float(sep.objVal)

        # Paso 6–9
        if obj_sep < obj_master - tol:
            Y.append(y_sol)
        else:
            optimal = True

    # ============================
    # SALIDA
    # ============================
    sol = {
        "a": np.array(a_hat),
        "b": b_hat,
        "c": c_sol,
        "x": x_sol
    }

    return {"status": GRB.OPTIMAL, "sol": sol, "iters": iteration}


### Variación de solo las restricciones

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np


def solve_problem_8(
    v,
    c_hat,
    a_hat,
    b_hat,
    A,
    b_vec,
    bounds,
    var_types,
    d_star,
    D_mat,
    d_vec,
    a_lb,
    a_ub,
    b_lb,
    b_ub,
    problem_type="min",
    tolerance=1e-6,
    max_iters=100000,
):
    """
    Algorithm 2 – Solve Problem (8)

    Returns
    -------
    a_sol : np.ndarray
    b_sol : float
    """

    n = len(c_hat)
    Y = []
    iteration = 0

    while iteration < max_iters:
        iteration += 1

        # ====================================================
        # MASTER PROBLEM
        # ====================================================
        master = gp.Model("master")
        master.Params.OutputFlag = 0

        # ----- x variables -----
        x = []
        for i in range(n):
            if var_types[i] == "C":
                x.append(master.addVar(lb=bounds[i][0], ub=bounds[i][1]))
            elif var_types[i] == "B":
                x.append(master.addVar(vtype=GRB.BINARY))
            else:
                x.append(master.addVar(lb=bounds[i][0], ub=bounds[i][1],
                                       vtype=GRB.INTEGER))

        # ----- a, b variables -----
        a = master.addVars(n, vtype=GRB.INTEGER, name="a")
        b = master.addVar(vtype=GRB.INTEGER, name="b")

        # ----- L1 auxiliaries -----
        u = master.addVars(n + 1, lb=0)

        for i in range(n):
            master.addConstr(a[i] - a_hat[i] <= u[i])
            master.addConstr(a_hat[i] - a[i] <= u[i])

        master.addConstr(b - b_hat <= u[n])
        master.addConstr(b_hat - b <= u[n])

        master.setObjective(gp.quicksum(u[i] for i in range(n + 1)),
                            GRB.MINIMIZE)

        # ----- cortes acumulados -----
        for y in Y:
            if problem_type == "min":
                master.addConstr(
                    gp.quicksum(a[i] * y[i] for i in range(n)) <= b - 1
                )
            else:
                master.addConstr(
                    gp.quicksum(a[i] * y[i] for i in range(n)) >= b + 1
                )

        # ----- fijar valor objetivo -----
        master.addConstr(
            gp.quicksum(c_hat[i] * x[i] for i in range(n)) == v
        )

        # ----- restricción clave -----
        if problem_type == "min":
            master.addConstr(gp.quicksum(a[i] * x[i] for i in range(n)) >= b)
        else:
            master.addConstr(gp.quicksum(a[i] * x[i] for i in range(n)) <= b)

        # ----- x ∈ X -----
        for i in range(len(b_vec)):
            master.addConstr(
                gp.quicksum(A[i, j] * x[j] for j in range(n)) >= b_vec[i]
            )

        # ----- x ∈ D (conjunto favorable) -----
        for i in range(len(d_vec)):
            master.addConstr(
                gp.quicksum(D_mat[i, j] * x[j] for j in range(n)) <= d_vec[i]
            )

        # ✅ CORRECCIÓN: presupuesto de distancia siempre presente
        # Si d_star es inf, Gurobi lo maneja correctamente
        if np.isfinite(d_star):
            master.addConstr(gp.quicksum(u[i] for i in range(n + 1)) <= d_star)

        # H_{a,b} = caja
        for i in range(n):
            master.addConstr(a[i] >= a_lb[i])
            master.addConstr(a[i] <= a_ub[i])

        master.addConstr(b >= b_lb)
        master.addConstr(b <= b_ub)

        master.optimize()


        if master.status != GRB.OPTIMAL:
            return None, None

        a_sol = np.array([a[i].X for i in range(n)])
        b_sol = b.X

                # ====================================================
        # SEPARATION 1: soluciones estrictamente mejores
        # ====================================================
        sep = gp.Model("sep_strict")
        sep.Params.OutputFlag = 0

        y = []
        for i in range(n):
            if var_types[i] == "C":
                y.append(sep.addVar(lb=bounds[i][0], ub=bounds[i][1]))
            elif var_types[i] == "B":
                y.append(sep.addVar(vtype=GRB.BINARY))
            else:
                y.append(sep.addVar(lb=bounds[i][0], ub=bounds[i][1],
                                   vtype=GRB.INTEGER))

        for i in range(len(b_vec)):
            sep.addConstr(
                gp.quicksum(A[i, j] * y[j] for j in range(n)) >= b_vec[i]
            )

        sep.addConstr(gp.quicksum(c_hat[i] * y[i] for i in range(n)) <= v - 1)
        sep.setObjective(gp.quicksum(a_sol[i] * y[i] for i in range(n)),
                         GRB.MAXIMIZE)
        sep.optimize()

        if sep.status == GRB.OPTIMAL:
            y_sol = np.array([y[i].X for i in range(n)])
            if np.dot(a_sol, y_sol) >= b_sol - tolerance:
                Y.append(y_sol)
                continue

                # ====================================================
        # CHECK (WCE): ¿existe al menos un óptimo en D con costo v?
        # ====================================================
        sep_in_D = gp.Model("sep_in_D")
        sep_in_D.Params.OutputFlag = 0

        y = []
        for i in range(n):
            if var_types[i] == "C":
                y.append(sep_in_D.addVar(lb=bounds[i][0], ub=bounds[i][1]))
            elif var_types[i] == "B":
                y.append(sep_in_D.addVar(vtype=GRB.BINARY))
            else:
                y.append(sep_in_D.addVar(lb=bounds[i][0], ub=bounds[i][1],
                                         vtype=GRB.INTEGER))

        # Restricciones X
        for i in range(len(b_vec)):
            sep_in_D.addConstr(gp.quicksum(A[i, j] * y[j] for j in range(n)) >= b_vec[i])

        # Restricciones D (y ∈ D)
        for i in range(len(d_vec)):
            sep_in_D.addConstr(gp.quicksum(D_mat[i, j] * y[j] for j in range(n)) <= d_vec[i] + 1e-9)

        # Igualdad de costo: c_hat^T y == v
        sep_in_D.addConstr(gp.quicksum(c_hat[i] * y[i] for i in range(n)) == v)

        # Solo buscamos factibilidad: si hay solución, WCE existe
        sep_in_D.setObjective(0.0, GRB.MINIMIZE)
        sep_in_D.optimize()

        if sep_in_D.Status == GRB.OPTIMAL:
            # ✅ Existe al menos un óptimo en D -> WCE satisfecho. Terminamos.
            break

        # Si llegamos aquí: NO existe y ∈ D con costo v.
        # Para forzar al master a moverse, buscamos cualquier y (empate) con costo v (puede estar fuera de D)
        # y lo agregamos a Y para que el master lo considere como 'caso a bloquear' para la (a,b) actual.
        # Esto NO persigue eliminar *todos* los empates fuera de D de forma permanente;
        # simplemente le da al master un punto concreto que lo obliga a cambiar.
        sep_any = gp.Model("sep_any_tied")
        sep_any.Params.OutputFlag = 0
        y = []
        for i in range(n):
            if var_types[i] == "C":
                y.append(sep_any.addVar(lb=bounds[i][0], ub=bounds[i][1]))
            elif var_types[i] == "B":
                y.append(sep_any.addVar(vtype=GRB.BINARY))
            else:
                y.append(sep_any.addVar(lb=bounds[i][0], ub=bounds[i][1],
                                        vtype=GRB.INTEGER))
        for i in range(len(b_vec)):
            sep_any.addConstr(gp.quicksum(A[i, j] * y[j] for j in range(n)) >= b_vec[i])
        sep_any.addConstr(gp.quicksum(c_hat[i] * y[i] for i in range(n)) == v)
        sep_any.setObjective(gp.quicksum(a_sol[i] * y[i] for i in range(n)), GRB.MAXIMIZE)
        sep_any.optimize()

        if sep_any.Status == GRB.OPTIMAL:
            y_sol = np.array([yi.X for yi in y])
            # Añadimos este y a Y para que master lo tenga en cuenta (forzar cambio en (a,b))
            Y.append(y_sol)
            continue

        # Si tampoco encontramos empates (sep_any no óptimo), no hay nada que bloquear: aceptamos (a,b)
        break

    return a_sol, b_sol

In [ ]:
import numpy as np
def compute_c_bounds(c_hat, A, b_vec, D_mat, d_vec, a_ub, b_lb, bounds, var_types, time_limit=5):
    """
    Calcula cotas c_min y c_max para el problema con manejo robusto de infeasibilidad.

    ✅ CORRECCIÓN: Si el problema con restricción a_max'x >= b_min es infeasible,
    intenta sin esa restricción y usa cotas muy conservadoras.
    """
    n = len(c_hat)
    a_max = a_ub
    b_min = b_lb

    def build_model(sense, include_ab_constraint=True):
        m = gp.Model()
        m.Params.OutputFlag = 0
        m.Params.TimeLimit = time_limit
        m.Params.MIPFocus = 1  # Enfocarse en encontrar soluciones factibles

        x = []
        for i in range(n):
            if var_types[i] == 'B':
                x.append(m.addVar(vtype=GRB.BINARY))
            elif var_types[i] == 'I':
                x.append(m.addVar(lb=bounds[i][0], ub=bounds[i][1], vtype=GRB.INTEGER))
            else:
                x.append(m.addVar(lb=bounds[i][0], ub=bounds[i][1]))

        # Restricciones A x >= b_vec
        for i in range(A.shape[0]):
            m.addConstr(gp.quicksum(A[i,j]*x[j] for j in range(n)) >= b_vec[i])

        # Restricciones D (conjunto favorable)
        for i in range(D_mat.shape[0]):
            m.addConstr(gp.quicksum(D_mat[i,j]*x[j] for j in range(n)) <= d_vec[i])

        # ✅ Restricción opcional a_max'x >= b_min
        if include_ab_constraint:
            m.addConstr(gp.quicksum(a_max[i]*x[i] for i in range(n)) >= b_min)

        m.setObjective(gp.quicksum(c_hat[i]*x[i] for i in range(n)), sense)
        return m

    # ========================================
    # Intento 1: Con restricción a_max'x >= b_min
    # ========================================
    try:
        m_min = build_model(GRB.MINIMIZE, include_ab_constraint=True)
        m_min.optimize()

        if m_min.status == GRB.OPTIMAL:
            c_min = m_min.ObjVal
        elif m_min.status == GRB.TIME_LIMIT and m_min.SolCount > 0:
            c_min = m_min.ObjVal
        else:
            raise RuntimeError("Infeasible with ab_constraint")

        m_max = build_model(GRB.MAXIMIZE, include_ab_constraint=True)
        m_max.optimize()

        if m_max.status == GRB.OPTIMAL:
            c_max = m_max.ObjVal
        elif m_max.status == GRB.TIME_LIMIT and m_max.SolCount > 0:
            c_max = m_max.ObjVal
        else:
            raise RuntimeError("Infeasible with ab_constraint")

        return int(np.floor(c_min)), int(np.ceil(c_max))

    except Exception as e:
        # Print the status when an error occurs
        status_min = m_min.status if 'm_min' in locals() else 'N/A'
        status_max = m_max.status if 'm_max' in locals() else 'N/A'
        print(f"compute_c_bounds (Exception attempt 1): Status m_min={status_min}, m_max={status_max}, Error: {e}")
        pass  # Continuar al intento 2

    # ========================================
    # Intento 2: SIN restricción a_max'x >= b_min
    # ========================================
    try:
        m_min = build_model(GRB.MINIMIZE, include_ab_constraint=False)
        m_min.optimize()

        if m_min.status not in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
            raise RuntimeError("compute_c_bounds: Problema base infeasible")

        c_min = m_min.ObjVal if m_min.status == GRB.OPTIMAL else m_min.ObjBound

        m_max = build_model(GRB.MAXIMIZE, include_ab_constraint=False)
        m_max.optimize()

        if m_max.status not in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
            raise RuntimeError("compute_c_bounds: Problema base infeasible")

        c_max = m_max.ObjVal if m_max.status == GRB.OPTIMAL else m_max.ObjBound

        # ✅ Expandir cotas conservadoramente
        margin = max(10, int(0.2 * abs(c_max - c_min)))
        return int(np.floor(c_min - margin)), int(np.ceil(c_max + margin))

    except Exception as e:
        status_min = m_min.status if 'm_min' in locals() else 'N/A'
        status_max = m_max.status if 'm_max' in locals() else 'N/A'
        raise RuntimeError(f"compute_c_bounds: Infeasible incluso sin restricción ab: {e}")


def compute_lower_bound_lemma4_L1(
    c_hat,
    v_bar,
    A,
    b_vec,
    bounds,
    var_types,
    a_hat,
    b_hat,
    a_lb,
    a_ub,
    b_lb,
    b_ub,
    time_limit,
    max_cuts = 100
):

    t0 = time.perf_counter()
    n = len(c_hat)
    p = len(a_hat)

    m = gp.Model()
    m.Params.OutputFlag = 0
    m.Params.TimeLimit = time_limit
    m.Params.MIPFocus = 1  # Enfocarse en encontrar soluciones factibles

    a = m.addVars(p, lb=a_lb, ub=a_ub, vtype=GRB.INTEGER)
    b = m.addVar(lb=b_lb, ub=b_ub, vtype=GRB.INTEGER)

    u = m.addVars(p, lb=0)
    v_aux = m.addVar(lb=0)

    for i in range(p):
        m.addConstr(a[i] - a_hat[i] <= u[i])
        m.addConstr(a_hat[i] - a[i] <= u[i])

    m.addConstr(b - b_hat <= v_aux)
    m.addConstr(b_hat - b <= v_aux)

    for i in range(p):
        m.addConstr(a[i] >= a_lb[i])
        m.addConstr(a[i] <= a_ub[i])

    m.addConstr(b >= b_lb)
    m.addConstr(b <= b_ub)


    m.setObjective(gp.quicksum(u[i] for i in range(p)) + v_aux, GRB.MINIMIZE)

    cuts = 0
    while cuts < max_cuts:
        if time.perf_counter() - t0 > time_limit:
            return np.inf

        m.optimize()
        if m.status != GRB.OPTIMAL:
            return np.inf

        a_val = np.array([a[i].X for i in range(p)])
        b_val = b.X

        sep = gp.Model()
        sep.Params.OutputFlag = 0
        sep.Params.TimeLimit = time_limit
        sep.Params.MIPFocus = 1  # Enfocarse en encontrar soluciones factibles

        y = []
        for i in range(n):
            if var_types[i] == 'B':
                y.append(sep.addVar(vtype=GRB.BINARY))
            elif var_types[i] == 'I':
                y.append(sep.addVar(lb=bounds[i][0], ub=bounds[i][1], vtype=GRB.INTEGER))
            else:
                y.append(sep.addVar(lb=bounds[i][0], ub=bounds[i][1]))

        for i in range(A.shape[0]):
            sep.addConstr(gp.quicksum(A[i,j]*y[j] for j in range(n)) >= b_vec[i])

        sep.addConstr(gp.quicksum(c_hat[i]*y[i] for i in range(n)) <= v_bar - 1)
        sep.setObjective(gp.quicksum(a_val[i]*y[i] for i in range(n)), GRB.MAXIMIZE)
        sep.optimize()

        if sep.status != GRB.OPTIMAL or sep.ObjVal <= b_val - 1 + 1e-6:
            break

        y_star = np.array([y[i].X for i in range(n)])
        m.addConstr(gp.quicksum(a[i]*y_star[i] for i in range(n)) <= b - 1)
        cuts += 1

    return m.ObjVal


def algorithm3(c_hat, a_hat, b_hat, A, b_vec, a_lb, a_ub, b_lb, b_ub, bounds, var_types, D_mat, d_vec, sense="min", time_limit_seconds=120):
    """
    Algorithm 3: Optimal Weak CE for Mutable Constraint Parameters
    """
    # Inicialización
    t0 = time.perf_counter()
    a_star_best = None
    b_star_best = None
    d_star = np.inf
    stopped_by_lemma4 = False # Initialize the flag

    # ❌ ELIMINAR ESTA LÍNEA - ES INCORRECTA
    # if sense == "min":
    #     c_hat = -c_hat

    # Calcular cotas de v
    c_min, c_max = compute_c_bounds(c_hat, A, b_vec, D_mat, d_vec, a_ub, b_lb, bounds, var_types, time_limit=5)


    # Iterar sobre todos los valores posibles de v
    for v in range(c_min, c_max + 1):
        # Calcular lower bound (Lemma 4)
        lb = compute_lower_bound_lemma4_L1(
            c_hat,
            v,
            A,
            b_vec,
            bounds,
            var_types,
            a_hat,
            b_hat,
            a_lb,
            a_ub,
            b_lb,
            b_ub,
            time_limit=30,
            max_cuts=500
        )
        # Si lb >= d_star actual → podemos **cortar** la iteración
        if a_star_best is not None and lb >= d_star:
            print("Stop early, Lemma 4 lower bound ≥ d_star")
            stopped_by_lemma4 = True
            break

        if (time.perf_counter() - t0) >= time_limit_seconds:
            print("⏱️ Time limit alcanzado en Algorithm 3")
            break

        # ✅ CORRECCIÓN: Pasar d_star siempre (inf en primera iteración)
        result = solve_problem_8(
            v=v,
            c_hat=c_hat,
            a_hat=a_hat,
            b_hat=b_hat,
            A=A,
            b_vec=b_vec,
            bounds=bounds,
            var_types=var_types,
            d_star=d_star,  # ✅ Pasar d_star directamente (puede ser inf)
            D_mat=D_mat,
            d_vec=d_vec,
            a_lb=a_lb,
            a_ub=a_ub,
            b_lb=b_lb,
            b_ub=b_ub,
            problem_type=sense  # ✅ Pasar "min" o "max" directamente
        )

        if result is None or result[0] is None or result[1] is None:
            continue

        a_star_v, b_star_v = result

        # Calcular distancia L1
        dist_v = np.sum(np.abs(a_star_v - a_hat)) + abs(b_star_v - b_hat)
        print(f"v={v}: Distancia obtenida: {dist_v}")

        # Actualizar mejor solución
        if dist_v < d_star:
            a_star_best = a_star_v
            b_star_best = b_star_v
            d_star = dist_v

    out = {
        "a_star_best": a_star_best,
        "b_star_best": b_star_best,
        "d_star": d_star,
        "c_min": c_min,
        "c_max": c_max,
        "iters": v - c_min + 1 if 'v' in locals() else 0,
        "stopped_by_lemma4": stopped_by_lemma4
    }
    return out

## Prueba del código

Para crear las instances, se tomaron ciertos valores de A, c y x y luego se escogió un valor de x para que sea valido

In [ ]:
def solve_P_choose_argmin_in_D(
    A, b, c_hat,
    bounds, var_types,
    favorable_constraints=None,
    problem_type="min",
    tol=1e-6
):
    """
    1) Resuelve el problema original sobre TODO X.
    2) Fija el valor óptimo global.
    3) Verifica si existe un óptimo que pertenezca al conjunto favorable.
    """

    n = len(c_hat)

    # ======================
    # FASE 1: ÓPTIMO GLOBAL
    # ======================
    m = gp.Model("P_global")
    m.Params.OutputFlag = 0

    x = []
    for i in range(n):
        lb, ub = bounds[i]
        if var_types[i] == 'B':
            x.append(m.addVar(vtype=GRB.BINARY))
        elif var_types[i] == 'I':
            x.append(m.addVar(lb=lb, ub=ub, vtype=GRB.INTEGER))
        else:
            x.append(m.addVar(lb=lb, ub=ub))

    for i in range(len(b)):
        m.addConstr(gp.quicksum(A[i, j] * x[j] for j in range(n)) >= b[i])

    obj = gp.quicksum(c_hat[i] * x[i] for i in range(n))
    m.setObjective(obj, GRB.MAXIMIZE if problem_type == 'max' else GRB.MINIMIZE)
    m.optimize()

    if m.status != GRB.OPTIMAL:
        return m.status, None, None

    opt_val = float(m.objVal)

    # ======================
    # FASE 2: ÓPTIMO EN D
    # ======================
    m.addConstr(obj == opt_val)

    if favorable_constraints:
        for fc in favorable_constraints:
            expr = gp.quicksum(fc['coeffs'][j] * x[j] for j in range(n))
            if fc['sense'] == '>=':
                m.addConstr(expr >= fc['rhs'])
            elif fc['sense'] == '<=':
                m.addConstr(expr <= fc['rhs'])
            else:
                m.addConstr(expr == fc['rhs'])

    m.setObjective(0, GRB.MINIMIZE)
    m.optimize()

    if m.status == GRB.INFEASIBLE:
        return GRB.INFEASIBLE, None, opt_val

    x_star = np.array([x[i].X for i in range(n)])
    return GRB.OPTIMAL, x_star, opt_val


def generate_hard_instance(n, rng, lb_range=(-100, 30), ub_range=(-30, 100)):
    """
    Genera una instancia factible de MILP con:
    - Todas las variables enteras.
    - Límites inferiores y superiores aleatorios.
    - Restricciones A x >= b calculadas desde una solución factible.
    """

    # ---------- Crear solución x factible ----------
    x_sol = np.zeros(n, dtype=int)
    bounds = []
    var_types = []
    var_names = []

    for i in range(n):
        # Límite inferior aleatorio dentro de lb_range
        lb = rng.integers(lb_range[0], lb_range[1]+1)
        # Límite superior aleatorio, asegurando que sea mayor que lb
        ub_min = max(lb + 1, ub_range[0])
        ub = rng.integers(ub_min, ub_range[1]+1)
        # Generar x_sol factible dentro de [lb, ub]
        x_sol[i] = rng.integers(lb, ub+1)

        bounds.append((lb, ub))
        var_types.append('I')
        var_names.append(f"x_{i}")

    # ---------- Costos ----------
    c_hat = rng.integers(-50, 50, size=n)
    c_signs = ['>='] * n  # todas enteras, para WCE podemos usar >=

    # ---------- Restricciones ----------
    m = max(1, n // 2)
    A = rng.integers(-50, 50, size=(m, n))

    # Asegurar que ninguna fila sea nula
    for i in range(m):
        if np.all(A[i] == 0):
            A[i, rng.integers(0, n)] = 1

    # Calcular b para garantizar factibilidad
    b = np.zeros(m)
    for i in range(m):
        slack = rng.integers(0, 5)
        b[i] = np.dot(A[i], x_sol) - slack# A x >= b

    # ---------- Conjunto favorable ----------
    n_fc = rng.integers(1, 4)  # por ejemplo 1 a 3 restricciones
    favorable_constraints = []

    for i_fc in range(n_fc):
        vars_in_constraint = rng.choice(n, size=rng.integers(1, n//2+1), replace=False)
        coeffs = [rng.integers(1, 5) if j in vars_in_constraint else 0 for j in range(n)]

        # rhs garantizado para que x_sol lo cumpla
        slack = rng.integers(0, 3)  # pequeño margen opcional
        rhs = int(np.dot(coeffs, x_sol) - slack)

        sense = ">="  # siempre ">=" para mantener factibilidad
        favorable_constraints.append({
            "coeffs": coeffs,
            "sense": sense,
            "rhs": rhs,
            "name": f"fc_{i_fc}"
        })

    return {
        "A": A,
        "b": b,
        "c_hat": c_hat,
        "bounds": bounds,
        "var_types": var_types,
        "var_names": var_names,
        "favorable_constraints": favorable_constraints,
        "c_signs": c_signs,
        "x_sol": x_sol
    }


def verify_algorithm1_solution(sol, A, b, bounds, var_types,
                               favorable_constraints=None,
                               tol=1e-6):
    """
    Verifica SOLO lo que garantiza Algorithm 1:
    - x̂ ∈ X ∩ D
    - x̂ es óptima global para c*
    """

    c_star = sol["c"]
    x_hat = sol["x"]
    n = len(x_hat)

    # 1) factibilidad primal
    if np.dot(A, x_hat).min() < b.min() - tol:
        return {"verified_ok": False, "reason": "infeasible x̂"}

    if favorable_constraints:
        for fc in favorable_constraints:
            lhs = np.dot(fc["coeffs"], x_hat)
            if fc["sense"] == ">=" and lhs < fc["rhs"] - tol:
                return {"verified_ok": False, "reason": "x̂ ∉ D"}
            if fc["sense"] == "<=" and lhs > fc["rhs"] + tol:
                return {"verified_ok": False, "reason": "x̂ ∉ D"}
            if fc["sense"] == "==" and abs(lhs - fc["rhs"]) > tol:
                return {"verified_ok": False, "reason": "x̂ ∉ D"}

    # 2) optimalidad global (separación)
    m = gp.Model()
    m.Params.OutputFlag = 0

    y = []
    for i in range(n):
        lb, ub = bounds[i]
        if var_types[i] == "B":
            y.append(m.addVar(vtype=GRB.BINARY))
        elif var_types[i] == "I":
            y.append(m.addVar(lb=lb, ub=ub, vtype=GRB.INTEGER))
        else:
            y.append(m.addVar(lb=lb, ub=ub))

    for i in range(len(b)):
        m.addConstr(gp.quicksum(A[i,j] * y[j] for j in range(n)) >= b[i])

    m.setObjective(gp.quicksum(c_star[i] * y[i] for i in range(n)), GRB.MINIMIZE)
    m.optimize()

    if m.objVal < np.dot(c_star, x_hat) - tol:
        return {"verified_ok": False, "reason": "not globally optimal"}

    return {"verified_ok": True}





In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

def verify_c_as_wce(
    c_new,         # vector de costos modificado
    x_star,        # solución candidata
    A, b_vec,      # restricciones originales
    D_mat, d_vec,  # restricciones adversarias
    bounds,
    var_types,
    tol=1e-6,
    verbose=False
):
    """
    Verifica si los costos c_new generan un WCE respecto a x_star
    """
    n = len(c_new)

    # ---------------------------
    # Paso 1: Evaluar factibilidad de x_star en restricciones originales
    # ---------------------------
    for i in range(len(A)):
        # Originalmente: A[i] @ x_star >= b_vec[i]
        # Verificar violación: A[i] @ x_star < b_vec[i]
        if np.dot(A[i], x_star) < b_vec[i] - tol:
            if verbose:
                print(f"❌ x_star viola restricción original {i}")
            return False

    # ---------------------------
    # Paso 2: Revisar restricciones adversarias (conjunto favorable)
    # ---------------------------
    for i in range(len(D_mat)):
        # Originalmente: D_mat[i] @ x_star >= d_vec[i]
        # Verificar violación: D_mat[i] @ x_star < d_vec[i]
        if np.dot(D_mat[i], x_star) < d_vec[i] - tol:
            if verbose:
                print(f"❌ x_star viola restricción adversaria {i}")
            return False

    # ---------------------------
    # Paso 3: Comprobar que x_star minimiza c_new^T x bajo las restricciones
    # ---------------------------
    m = gp.Model()
    m.Params.OutputFlag = 0

    x = []
    for i in range(n):
        lb, ub = bounds[i]
        vtype = var_types[i]
        if vtype == "B":
            x.append(m.addVar(vtype=GRB.BINARY, lb=lb, ub=ub))
        elif vtype == "I":
            x.append(m.addVar(vtype=GRB.INTEGER, lb=lb, ub=ub))
        else:
            x.append(m.addVar(lb=lb, ub=ub))

    # Restricciones originales (A @ x >= b_vec)
    for i in range(len(A)):
        m.addConstr(gp.quicksum(A[i, j]*x[j] for j in range(n)) >= b_vec[i])

    # Restricciones adversarias (D_mat @ x >= d_vec)
    for i in range(len(D_mat)):
        m.addConstr(gp.quicksum(D_mat[i, j]*x[j] for j in range(n)) >= d_vec[i])

    # Objetivo: minimizar c_new^T x
    m.setObjective(gp.quicksum(c_new[j]*x[j] for j in range(n)), GRB.MINIMIZE)
    m.optimize()

    if m.Status != GRB.OPTIMAL:
        if verbose:
            print("❌ El solver no encontró óptimo")
        return False

    obj_val = m.ObjVal
    obj_x_star = np.dot(c_new, x_star)

    # ---------------------------
    # Paso 4: Verificar WCE
    # ---------------------------
    if obj_x_star - obj_val > tol:
        if verbose:
            print(f"❌ x_star no es óptimo para c_new: {obj_x_star} > {obj_val}")
        return False

    if verbose:
        print(f"✅ x_star es WCE para c_new, obj={obj_x_star}")

    return True


In [ ]:
def run_one_trial_algorithm1(n, seed, alg_time_limit=120):
    rng = np.random.default_rng(seed)

    inst = generate_hard_instance(n, rng)

    globals()['A'] = inst["A"]
    globals()['b'] = inst["b"]

    t0 = time.perf_counter()

    res = algorithm1(
    a_hat=None,
    b_hat=None,
    c_hat=inst["c_hat"],
    H_bounds=None,
    M=None,
    tol=1e-6,
    time_limit_seconds=alg_time_limit,
    favorable_constraints=inst["favorable_constraints"],
    c_signs=inst["c_signs"],
    bounds=inst["bounds"],
    var_types=inst["var_types"]
)


    runtime = time.perf_counter() - t0

    out = {
    "seed": seed,
    "n": n,
    "runtime_sec": runtime,
    "time_limit_sec": alg_time_limit,
    "status": res["status"],
    "iters": res["iters"],
    "dist_L1": None,              # 👈 NUEVO
    "verified_ok": False
    }

    if res["status"] == GRB.OPTIMAL:
        c_sol = res["sol"]["c"]
        x_sol = res["sol"]["x"]

        # --- Calcular L1 distance ---
        out["dist_L1"] = float(np.sum(np.abs(c_sol - inst["c_hat"])))

        # --- Construir D_mat y d_vec desde favorable_constraints ---
        n = len(c_sol)
        D_mat = []
        d_vec = []

        for fc in inst["favorable_constraints"]:
            D_mat.append(fc["coeffs"])
            d_vec.append(fc["rhs"])
        D_mat = np.array(D_mat)
        d_vec = np.array(d_vec)

        # --- Verificar que c_sol genere efectivamente un WCE ---
        verified_wce = verify_c_as_wce(
            c_new=c_sol,
            x_star=x_sol,
            A=inst["A"],
            b_vec=inst["b"],
            D_mat=D_mat,
            d_vec=d_vec,
            bounds=inst["bounds"],
            var_types=inst["var_types"],
            tol=1e-6,
            verbose=False
        )

        out["verified_ok"] = verified_wce

        # --- Opcional: resultados de sanity check original ---
        """v = verify_algorithm1_solution(
            res["sol"],
            inst["A"], inst["b"],
            inst["bounds"], inst["var_types"],
            inst["favorable_constraints"]
        )
        out.update(v)"""
    return out




def benchmark_algorithm1(ns, trials_per_n=10, base_seed=100):
    rows = []
    seed = base_seed

    total_trials = len(ns) * trials_per_n
    completed = 0

    for n in ns:
        for _ in range(trials_per_n):
            rows.append(run_one_trial_algorithm1(n, seed))
            seed += 1
            completed += 1

            # Mostrar progreso cada iteración
            progress = completed / total_trials
            bar_length = 40
            filled_length = int(bar_length * progress)
            bar = "█" * filled_length + "-" * (bar_length - filled_length)
            print(f"\rProgreso: |{bar}| {progress*100:.1f}% ({completed}/{total_trials})", end="")

    print()  # Salto de línea al final
    return pd.DataFrame(rows)


In [ ]:
# ---------------------------------------
# Stats + ajuste Normal + test (opcional)
# ---------------------------------------
def summarize(df):
    n_total = len(df)
    # Modificado para considerar varios estados de 'OPTIMAL'
    n_opt = int((df["status"].isin(["OPTIMAL", "OPTIMAL_WCE"])).sum())
    n_tl = int((df["status"] == "TIME_LIMIT").sum())
    p_tl = n_tl / n_total if n_total else float("nan")

    finished = df[~df["status"].isin(["TIME_LIMIT", "ALGORITHM_ERROR", "GENERATION_ERROR", "INSTANCE_INFEASIBLE"])]["runtime_sec"].to_numpy()
    mean_finished = float(np.mean(finished)) if len(finished) else float("nan")
    std_finished = float(np.std(finished, ddof=1)) if len(finished) >= 2 else float("nan")

    # Asegurar que 'time_limit_sec' se trata como un escalar
    tl = float(df["time_limit_sec"].iloc[0]) if not df.empty and "time_limit_sec" in df.columns else np.nan

    censored = df["runtime_sec"].to_numpy().copy()
    # Censorizar solo los que realmente llegaron al límite de tiempo
    censored[df["status"].to_numpy() == "TIME_LIMIT"] = tl
    mean_cens = float(np.mean(censored)) if len(censored) else np.nan
    std_cens = float(np.std(censored, ddof=1)) if len(censored) >= 2 else np.nan

    return pd.DataFrame([{
        "count_total": n_total,
        "count_optimal": n_opt,
        "count_time_limit": n_tl,
        "pct_time_limit": p_tl,
        "mean_finished_sec": mean_finished,
        "std_finished_sec": std_finished,
        "mean_censored_sec": mean_cens,
        "std_censored_sec": std_cens
    }])

# ---------------------------------------
# Gráficos de tiempo y L1 + tiempo por n
# ---------------------------------------
def plot_runtime(df):
    import matplotlib.pyplot as plt

    # Histograma de tiempos
    if not df.empty:
        plt.figure()
        plt.hist(df["runtime_sec"], bins=20, color='skyblue', edgecolor='black')
        plt.xlabel("Tiempo (s)")
        plt.ylabel("Frecuencia")
        plt.title("Distribución de tiempos de ejecución")
        plt.grid(True)
        plt.show()
    else:
        print("DataFrame vacío, no se puede graficar el histograma de tiempos.")

    # Histograma de distancias L1
    # Convertir a numérico y eliminar NaNs para asegurar compatibilidad con np.isfinite
    dist = pd.to_numeric(df["dist_L1"], errors='coerce').dropna()

    if len(dist) > 0:
        plt.figure()
        plt.hist(dist, bins=20, edgecolor='black')
        plt.xlabel("Distancia L1")
        plt.ylabel("Frecuencia")
        plt.title("Distribución de dist_L1 (valores finitos)")
        plt.grid(True)
        plt.show()
    else:
        print("No hay valores finitos de dist_L1 para graficar.")

    # Promedio y desviación por número de variables n
    if not df.empty:
        grouped = df.groupby("n")["runtime_sec"].agg(["mean", "std"]).reset_index()

        plt.figure()
        plt.errorbar(
            grouped["n"], grouped["mean"], yerr=grouped["std"],
            fmt='o-', ecolor='red', capsize=5, markersize=6, color='blue'
        )
        plt.xlabel("Número de variables n")
        plt.ylabel("Tiempo promedio (s)")
        plt.yscale("log")  # <-- eje y en escala logarítmica
        plt.title("Tiempo promedio de ejecución ± desviación estándar")
        plt.grid(True)
        plt.show()
    else:
        print("DataFrame vacío, no se puede graficar el tiempo promedio por n.")

In [ ]:
ns = [2, 4]
df = benchmark_algorithm1(ns, trials_per_n=50)

print(df.groupby(["n", "status", "verified_ok"]).size())
print(df.describe())
summary = summarize(df)
print(summary)
plot_runtime(df)

## Prueba algoritmo 3

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

def verify_algorithm3_solution(
    a_star,
    b_star,
    c_hat,
    A,
    b_vec,
    D_mat,
    d_vec,
    bounds,
    var_types,
    tol=1e-6,
    verbose=False
):
    """
    Verifica si (a*, b*) es un Weak Counterfactual Explanation válido.
    Maneja correctamente el caso de óptimos múltiples.
    """

    n = len(a_star)

    # ---------------------------
    # Paso 1: valor óptimo global
    # ---------------------------
    m1 = gp.Model()
    m1.Params.OutputFlag = 0

    x1 = []
    for i in range(n):
        lb, ub = bounds[i]
        vtype_str = var_types[i] # Get string type (e.g., 'I', 'B', 'C')
        if vtype_str == "B":
            vtype_grb = GRB.BINARY
        elif vtype_str == "I":
            vtype_grb = GRB.INTEGER
        else:
            vtype_grb = GRB.CONTINUOUS
        x1.append(m1.addVar(lb=lb, ub=ub, vtype=vtype_grb))

    # !!! CORRECCIÓN: Usar >= para las restricciones A @ x >= b_vec
    for i in range(len(A)):
        m1.addConstr(
            gp.quicksum(A[i][j] * x1[j] for j in range(n)) >= b_vec[i]
        )

    m1.addConstr(
        gp.quicksum(a_star[j] * x1[j] for j in range(n)) >= b_star
    )

    m1.setObjective(
        gp.quicksum(c_hat[j] * x1[j] for j in range(n)),
        GRB.MINIMIZE
    )

    m1.optimize()

    if m1.Status != GRB.OPTIMAL:
        return {
            "verified_ok": False,
            "reason": "Modified problem infeasible",
            "status": m1.Status
        }

    z_star = m1.ObjVal

    # ---------------------------------
    # Paso 2: ¿existe óptimo favorable?
    # ---------------------------------
    m2 = gp.Model()
    m2.Params.OutputFlag = 0

    x2 = []
    for i in range(n):
        lb, ub = bounds[i]
        vtype_str = var_types[i] # Get string type (e.g., 'I', 'B', 'C')
        if vtype_str == "B":
            vtype_grb = GRB.BINARY
        elif vtype_str == "I":
            vtype_grb = GRB.INTEGER
        else:
            vtype_grb = GRB.CONTINUOUS
        x2.append(m2.addVar(lb=lb, ub=ub, vtype=vtype_grb))

    # !!! CORRECCIÓN: Usar >= para las restricciones A @ x >= b_vec
    for i in range(len(A)):
        m2.addConstr(
            gp.quicksum(A[i][j] * x2[j] for j in range(n)) >= b_vec[i]
        )

    for i in range(len(D_mat)):
        m2.addConstr(
            gp.quicksum(D_mat[i][j] * x2[j] for j in range(n)) <= d_vec[i]
        )

    m2.addConstr(
        gp.quicksum(a_star[j] * x2[j] for j in range(n)) >= b_star
    )

    m2.addConstr(
        gp.quicksum(c_hat[j] * x2[j] for j in range(n)) <= z_star + tol
    )

    m2.setObjective(0.0, GRB.MINIMIZE)
    m2.optimize()

    verified_ok = (m2.Status == GRB.OPTIMAL)

    if verbose:
        if verified_ok:
            print("✅ Existe al menos un óptimo favorable → WCE válido")
        else:
            print("❌ Ningún óptimo favorable → NO es WCE")

    return {
        "verified_ok": verified_ok,
        "optimal_value": z_star,
        "status": m2.Status
    }


In [ ]:
def verify_instance_feasibility_alg3(inst):
    """
    Verifica que X ∩ D ≠ ∅ usando D_mat, d_vec directamente
    """
    n = len(inst["c_hat"])

    m = gp.Model()
    m.Params.OutputFlag = 0

    x = [
        m.addVar(
            lb=inst["bounds"][i][0],
            ub=inst["bounds"][i][1],
            vtype=GRB.CONTINUOUS
        )
        for i in range(n)
    ]

    # A x >= b
    for i in range(len(inst["b"])):
        m.addConstr(
            gp.quicksum(inst["A"][i, j] * x[j] for j in range(n)) >= inst["b"][i]
        )

    # D_mat x <= d_vec
    for i in range(len(inst["d_vec"])):
        m.addConstr(
            gp.quicksum(inst["D_mat"][i, j] * x[j] for j in range(n)) <= inst["d_vec"][i]
        )

    m.setObjective(0, GRB.MINIMIZE)
    m.optimize()

    if m.status == GRB.OPTIMAL:
        return True, np.array([x[i].X for i in range(n)])
    return False, None


In [ ]:
def generate_hard_instance3(n, rng):
    """
    Genera una instancia GARANTIZADA para Algorithm 3:
    - X ∩ D ≠ ∅ (AMPLIO)
    - existe conflicto en costo
    - (a_hat, b_hat) no separa (pero está cerca)
    - existe (a_star, b_star) cercano que sí separa
    """

    # =========================
    # 1. Dominio de variables
    # =========================
    bounds = [(-10, 10)] * n
    var_types = ["I"] * n

    # =========================
    # 2. Plano separador VERDADERO
    # =========================
    a_star = rng.integers(-1, 2, size=n) # Reduced range
    while np.all(a_star == 0):
        a_star = rng.integers(-1, 2, size=n) # Reduced range

    b_star = rng.integers(5, 10)

    # =========================
    # 3. Puntos positivos (en D)
    # =========================
    X_pos = []
    attempts = 0
    while len(X_pos) < 5 and attempts < 100:  # ✅ MÁS puntos
        x = rng.integers(-10, 11, size=n)
        if a_star @ x >= b_star + 1:
            X_pos.append(x)
        attempts += 1

    if len(X_pos) < 3:
        raise RuntimeError("No se pudieron generar suficientes puntos positivos")

    # =========================
    # 4. Puntos negativos (fuera de D)
    # =========================
    X_neg = []
    attempts = 0
    while len(X_neg) < 5 and attempts < 100:  # ✅ MÁS puntos
        x = rng.integers(-10, 11, size=n)
        if a_star @ x <= b_star - 2:
            X_neg.append(x)
        attempts += 1

    if len(X_neg) < 3:
        raise RuntimeError("No se pudieron generar suficientes puntos negativos")

    # =========================
    # 5. Restricciones A x >= b (envuelven X_pos ∪ X_neg)
    # =========================
    m_val = max(3, n // 2)  # ✅ MÁS restricciones
    A = []
    b_vec = []

    for _ in range(m_val):
        Ai = rng.integers(-1, 2, size=n)  # Reduced range
        if np.all(Ai == 0):
            Ai[rng.integers(0, n)] = 1

        # ✅ Agregar margen
        bi = min(Ai @ x for x in X_pos + X_neg) - rng.integers(5, 11)
        A.append(Ai)
        b_vec.append(bi)

    A = np.array(A)
    b_vec = np.array(b_vec)

    # =========================
    # 6. Conjunto favorable D (MUY RELAJADO)
    # =========================
    D_mat = []
    d_vec = []

    idx = rng.integers(0, n)
    coeff = np.zeros(n)
    coeff[idx] = 1

    # ✅ MARGEN ENORME: -50 a -100
    rhs = min(x[idx] for x in X_pos) - rng.integers(100, 201) # Increased margin
    D_mat.append(-coeff)
    d_vec.append(-rhs)

    D_mat = np.array(D_mat)
    d_vec = np.array(d_vec)

    # =========================
    # 7. Costos (garantizar conflicto)
    # =========================
    c_hat = rng.integers(1, 2, size=n) # Reduced range

    # ✅ Verificar y ajustar conflicto
    c_vals_pos = [c_hat @ x for x in X_pos]
    c_vals_neg = [c_hat @ x for x in X_neg]

    if min(c_vals_neg) >= min(c_vals_pos):
        # Ajustar c_hat para crear conflicto
        idx_best_neg = np.argmin(c_vals_neg)
        x_best_neg = X_neg[idx_best_neg]

        # Hacer que x_best_neg tenga costo menor
        for i in range(n):
            if x_best_neg[i] < 0:
                c_hat[i] += rng.integers(1, 3)

    # =========================
    # 8. Plano inicial perturbado (NO separador, pero CERCA)
    # =========================
    a_hat = a_star + rng.integers(-1, 2, size=n)
    b_hat = b_star + rng.integers(-1, 2)

    # ✅ Verificar si a_hat separa
    separates_pos = all(a_hat @ x >= b_hat for x in X_pos)
    separates_neg = all(a_hat @ x <= b_hat - 1 for x in X_neg)

    # Si separa, perturbar LEVEMENTE
    if separates_pos and separates_neg:
        b_hat += rng.integers(1, 3)  # Solo aumentar un poco

    # =========================
    # 9. Caja H_ab (PEQUEÑA)
    # =========================
    radius = 10  # Increased radius
    a_lb = a_hat - radius
    a_ub = a_hat + radius
    b_lb = b_hat - radius
    b_ub = b_hat + radius

    # =========================
    # 10. VERIFICACIÓN FINAL
    # =========================
    # Verificar que X ∩ D no está vacío
    m_test = gp.Model()
    m_test.Params.OutputFlag = 0

    x_test = [m_test.addVar(lb=-10, ub=10, vtype=GRB.INTEGER) for _ in range(n)]

    for i in range(len(b_vec)):
        m_test.addConstr(gp.quicksum(A[i,j] * x_test[j] for j in range(n)) >= b_vec[i])

    for i in range(len(d_vec)):
        m_test.addConstr(gp.quicksum(D_mat[i,j] * x_test[j] for j in range(n)) <= d_vec[i])

    m_test.optimize()
    print(f"generate_hard_instance3 (m_test): Gurobi status = {m_test.status}")

    if m_test.status != GRB.OPTIMAL:
        raise RuntimeError("X ∩ D está vacío")

    return {
        "A": A,
        "b": b_vec,
        "D_mat": D_mat,
        "d_vec": d_vec,
        "c_hat": c_hat,
        "bounds": bounds,
        "var_types": var_types,

        "a_hat": a_hat,
        "b_hat": b_hat,
        "a_lb": a_lb,
        "a_ub": a_ub,
        "b_lb": b_lb,
        "b_ub": b_ub,

        # Debug / ground truth
        "a_star": a_star,
        "b_star": b_star,
        "X_pos": X_pos,
        "X_neg": X_neg,
  }

In [ ]:
def run_one_trial_algorithm3_matrix(n, seed, alg_time_limit=120):
    rng = np.random.default_rng(seed)

    # =========================
    # 1. Generar instancia
    # =========================
    try:
        inst = generate_hard_instance3(n=n, rng=rng)
    except Exception as e:
        return {
            "seed": seed,
            "n": n,
            "status": "GENERATION_ERROR",
            "runtime_sec": 0.0,
            "iters": 0,
            "verified_ok": False,
            "error": str(e)[:80],
        }

    # =========================
    # 2. Verificar X ∩ D ≠ ∅
    # =========================
    is_feas, _ = verify_instance_feasibility_alg3(inst)
    if not is_feas:
        return {
            "seed": seed,
            "n": n,
            "status": "INSTANCE_INFEASIBLE",
            "runtime_sec": 0.0,
            "iters": 0,
            "verified_ok": False,
        }

    # =========================
    # 3. Ejecutar Algorithm 3
    # =========================
    t0 = time.perf_counter()
    try:
        res = algorithm3(
            c_hat=inst["c_hat"],
            a_hat=inst["a_hat"],
            b_hat=inst["b_hat"],
            A=inst["A"],
            b_vec=inst["b"],
            a_lb=inst["a_lb"],
            a_ub=inst["a_ub"],
            b_lb=inst["b_lb"],
            b_ub=inst["b_ub"],
            bounds=inst["bounds"],
            var_types=inst["var_types"],
            D_mat=inst["D_mat"],
            d_vec=inst["d_vec"],
            sense="min",
            time_limit_seconds=alg_time_limit,
        )
    except Exception as e:
        return {
            "seed": seed,
            "n": n,
            "status": "ALGORITHM_ERROR",
            "runtime_sec": time.perf_counter() - t0,
            "iters": 0,
            "verified_ok": False,
            "error": str(e)[:80],
        }

    runtime = time.perf_counter() - t0

    # =========================
    # 4. Interpretar resultado del algoritmo
    # =========================
    a_star = res.get("a_star_best")
    b_star = res.get("b_star_best")

    if a_star is not None and b_star is not None:
        dist_L1 = float(
            np.sum(np.abs(a_star - inst["a_hat"]))
            + abs(b_star - inst["b_hat"])
        )
        algo_status = "OPTIMAL"

    elif res.get("stopped_by_lemma4", False):
        dist_L1 = None
        algo_status = "NO_IMPROVEMENT"

    else:
        return {
            "seed": seed,
            "n": n,
            "runtime_sec": runtime,
            "time_limit_sec": alg_time_limit,
            "status": "INFEASIBLE",
            "iters": res.get("iters", 0),
            "d_star": None,
            "dist_L1": None,
            "verified_ok": False,
        }

    # =========================
    # 5. Verificación WCE CORRECTA
    # =========================
    ver = verify_algorithm3_solution(
        a_star=a_star,
        b_star=b_star,
        c_hat=inst["c_hat"],
        A=inst["A"],
        b_vec=inst["b"],
        D_mat=inst["D_mat"],
        d_vec=inst["d_vec"],
        bounds=inst["bounds"],
        var_types=inst["var_types"],
        tol=1e-6,
        verbose=False,
    )

    verified_ok = ver["verified_ok"]

    final_status = (
        "OPTIMAL_WCE" if verified_ok else "OPTIMAL_NOT_WCE"
        if algo_status == "OPTIMAL"
        else algo_status
    )

    # =========================
    # 6. Salida final
    # =========================
    return {
        "seed": seed,
        "n": n,
        "runtime_sec": runtime,
        "time_limit_sec": alg_time_limit,
        "status": final_status,
        "iters": res.get("iters", 0),
        "d_star": None if res.get("d_star", np.inf) == np.inf else res.get("d_star"),
        "dist_L1": dist_L1,
        "verified_ok": verified_ok,
    }


def benchmark_algorithm3_matrix(ns, trials_per_n=10, base_seed=200):
    """Igual que antes pero para Algorithm 3"""
    rows = []
    seed = base_seed

    total_trials = len(ns) * trials_per_n
    completed = 0

    for n in ns:
        for _ in range(trials_per_n):
            rows.append(run_one_trial_algorithm3_matrix(n, seed))
            seed += 1
            completed += 1

            progress = completed / total_trials
            bar_length = 40
            filled_length = int(bar_length * progress)
            bar = "█" * filled_length + "-" * (bar_length - filled_length)
            print(f"\rProgreso: |{bar}| {progress*100:.1f}% ({completed}/{total_trials})", end="")

    print()
    return pd.DataFrame(rows)


In [ ]:
# ---------------------------------------
# Stats + ajuste Normal + test (opcional)
# ---------------------------------------
def summarize(df):
    n_total = len(df)
    # Modificado para considerar varios estados de 'OPTIMAL'
    n_opt = int((df["status"].isin(["OPTIMAL", "OPTIMAL_WCE"])).sum())
    n_tl = int((df["status"] == "TIME_LIMIT").sum())
    p_tl = n_tl / n_total if n_total else float("nan")

    finished = df[~df["status"].isin(["TIME_LIMIT", "ALGORITHM_ERROR", "GENERATION_ERROR", "INSTANCE_INFEASIBLE"])]["runtime_sec"].to_numpy()
    mean_finished = float(np.mean(finished)) if len(finished) else float("nan")
    std_finished = float(np.std(finished, ddof=1)) if len(finished) >= 2 else float("nan")

    # Asegurar que 'time_limit_sec' se trata como un escalar
    tl = float(df["time_limit_sec"].iloc[0]) if not df.empty and "time_limit_sec" in df.columns else np.nan

    censored = df["runtime_sec"].to_numpy().copy()
    # Censorizar solo los que realmente llegaron al límite de tiempo
    censored[df["status"].to_numpy() == "TIME_LIMIT"] = tl
    mean_cens = float(np.mean(censored)) if len(censored) else np.nan
    std_cens = float(np.std(censored, ddof=1)) if len(censored) >= 2 else np.nan

    return pd.DataFrame([{
        "count_total": n_total,
        "count_optimal": n_opt,
        "count_time_limit": n_tl,
        "pct_time_limit": p_tl,
        "mean_finished_sec": mean_finished,
        "std_finished_sec": std_finished,
        "mean_censored_sec": mean_cens,
        "std_censored_sec": std_cens
    }])

# ---------------------------------------
# Gráficos de tiempo y L1 + tiempo por n
# ---------------------------------------
def plot_runtime(df):
    import matplotlib.pyplot as plt

    # Histograma de tiempos
    if not df.empty:
        plt.figure()
        plt.hist(df["runtime_sec"], bins=20, color='skyblue', edgecolor='black')
        plt.xlabel("Tiempo (s)")
        plt.ylabel("Frecuencia")
        plt.title("Distribución de tiempos de ejecución")
        plt.grid(True)
        plt.show()
    else:
        print("DataFrame vacío, no se puede graficar el histograma de tiempos.")

    # Histograma de distancias L1
    # Convertir a numérico y eliminar NaNs para asegurar compatibilidad con np.isfinite
    dist = pd.to_numeric(df["dist_L1"], errors='coerce').dropna()

    if len(dist) > 0:
        plt.figure()
        plt.hist(dist, bins=20, edgecolor='black')
        plt.xlabel("Distancia L1")
        plt.ylabel("Frecuencia")
        plt.title("Distribución de dist_L1 (valores finitos)")
        plt.grid(True)
        plt.show()
    else:
        print("No hay valores finitos de dist_L1 para graficar.")

    # Promedio y desviación por número de variables n
    if not df.empty:
        grouped = df.groupby("n")["runtime_sec"].agg(["mean", "std"]).reset_index()

        plt.figure()
        plt.errorbar(
            grouped["n"], grouped["mean"], yerr=grouped["std"],
            fmt='o-', ecolor='red', capsize=5, markersize=6, color='blue'
        )
        plt.xlabel("Número de variables n")
        plt.ylabel("Tiempo promedio (s)")
        plt.yscale("log")  # <-- eje y en escala logarítmica
        plt.title("Tiempo promedio de ejecución ± desviación estándar")
        plt.grid(True)
        plt.show()
    else:
        print("DataFrame vacío, no se puede graficar el tiempo promedio por n.")

In [ ]:
gp.setParam("Threads", 1)        # reproducibilidad + menos overhead
gp.setParam("OutputFlag", 0)
gp.setParam("MIPGap", 1e-6)
gp.setParam("Cuts", 1)
gp.setParam("Heuristics", 0.3)


In [ ]:
ns = [2, 4]
df = benchmark_algorithm3_matrix(ns, trials_per_n=50)

print(df.groupby(["n", "status", "verified_ok"]).size())
print(df.describe())
summary = summarize(df)
print(summary)
df_opt = df[df["status"] == "OPTIMAL_WCE"].reset_index(drop=True)
plot_runtime(df_opt)

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# --- Datos del ejemplo ---
c_hat = np.array([10, 20, 15])        # costos nominales
a_star = np.array([1, 1, 0])
b_star = 1
A = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
])                                     # restricciones x_i <= 1
b_vec = np.array([1, 1, 1])

D_mat = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
])                                     # restricciones adversarias simples
d_vec = np.array([1, 1, 1])

bounds = [(0, 1), (0, 1), (0, 1)]
var_types = [GRB.BINARY, GRB.BINARY, GRB.BINARY]  # variables binarias

# --- Solución candidata factible ---
# x_star cumple todas las restricciones y debería ser WCE
x_star = np.array([1, 0, 1])

# --- Llamada al sanity check ---
result = verify_algorithm3_solution(
    a_star=a_star,
    b_star=b_star,
    c_hat=c_hat,
    A=A,
    b_vec=b_vec,
    D_mat=D_mat,
    d_vec=d_vec,
    bounds=bounds,
    var_types=var_types,
    tol=1e-6,
    verbose=True
)

print(result)


In [ ]:
n = 2

bounds = [(0, 5), (0, 5)]
var_types = ["I", "I"]
A = np.array([
    [1, 0],
    [0, 1],
    [1, 1]
], dtype=int)

b_vec = np.array([0, 0, 3], dtype=int)
c_hat = np.array([1, 1], dtype=int)
a_hat = np.array([1, 1], dtype=int)
b_hat = 3
ab_radius = 2

a_lb = a_hat
a_ub = (1 + ab_radius) * a_hat   # [3, 3]

b_lb = b_hat
b_ub = (1 + ab_radius) * b_hat   # 9
D_mat = np.array([[1, 0]], dtype=int)
d_vec = np.array([1], dtype=int)
res = algorithm3(
    c_hat=c_hat,
    a_hat=a_hat,
    b_hat=b_hat,
    A=A,
    b_vec=b_vec,
    a_lb=a_lb,
    a_ub=a_ub,
    b_lb=b_lb,
    b_ub=b_ub,
    bounds=bounds,
    var_types=var_types,
    D_mat=D_mat,
    d_vec=d_vec,
    sense="min",
    time_limit_seconds=60
)
assert res["a_star_best"] is not None
assert res["b_star_best"] is not None
assert res["d_star"] >= 0
print("a* =", res["a_star_best"])
print("b* =", res["b_star_best"])
print("distancia L1 =", res["d_star"])



# Prueba con problema knapsack

### Predefinidas

In [ ]:
def verify_algorithm3_solution(
    a_star,
    b_star,
    c_hat,
    A,
    b_vec,
    D_mat,
    d_vec,
    bounds,
    var_types,
    tol=1e-6,
    verbose=False
):
    """
    Verifica si (a*, b*) es un Weak Counterfactual Explanation válido.
    Maneja correctamente el caso de óptimos múltiples.
    """

    n = len(a_star)

    # ---------------------------
    # Paso 1: valor óptimo global
    # ---------------------------
    m1 = gp.Model()
    m1.Params.OutputFlag = 0

    x1 = []
    for i in range(n):
        lb, ub = bounds[i]
        vtype_str = var_types[i] # Get string type (e.g., 'I', 'B', 'C')
        if vtype_str == "B":
            vtype_grb = GRB.BINARY
        elif vtype_str == "I":
            vtype_grb = GRB.INTEGER
        else:
            vtype_grb = GRB.CONTINUOUS
        x1.append(m1.addVar(lb=lb, ub=ub, vtype=vtype_grb))

    # !!! CORRECCIÓN: Usar >= para las restricciones A @ x >= b_vec
    for i in range(len(A)):
        m1.addConstr(
            gp.quicksum(A[i][j] * x1[j] for j in range(n)) >= b_vec[i]
        )

    m1.addConstr(
        gp.quicksum(a_star[j] * x1[j] for j in range(n)) >= b_star
    )

    m1.setObjective(
        gp.quicksum(c_hat[j] * x1[j] for j in range(n)),
        GRB.MINIMIZE
    )

    m1.optimize()

    if m1.Status != GRB.OPTIMAL:
        return {
            "verified_ok": False,
            "reason": "Modified problem infeasible",
            "status": m1.Status
        }

    z_star = m1.ObjVal

    # ---------------------------------
    # Paso 2: ¿existe óptimo favorable?
    # ---------------------------------
    m2 = gp.Model()
    m2.Params.OutputFlag = 0

    x2 = []
    for i in range(n):
        lb, ub = bounds[i]
        vtype_str = var_types[i] # Get string type (e.g., 'I', 'B', 'C')
        if vtype_str == "B":
            vtype_grb = GRB.BINARY
        elif vtype_str == "I":
            vtype_grb = GRB.INTEGER
        else:
            vtype_grb = GRB.CONTINUOUS
        x2.append(m2.addVar(lb=lb, ub=ub, vtype=vtype_grb))

    # !!! CORRECCIÓN: Usar >= para las restricciones A @ x >= b_vec
    for i in range(len(A)):
        m2.addConstr(
            gp.quicksum(A[i][j] * x2[j] for j in range(n)) >= b_vec[i]
        )

    for i in range(len(D_mat)):
        m2.addConstr(
            gp.quicksum(D_mat[i][j] * x2[j] for j in range(n)) <= d_vec[i]
        )

    m2.addConstr(
        gp.quicksum(a_star[j] * x2[j] for j in range(n)) >= b_star
    )

    m2.addConstr(
        gp.quicksum(c_hat[j] * x2[j] for j in range(n)) <= z_star + tol
    )

    m2.setObjective(0.0, GRB.MINIMIZE)
    m2.optimize()

    verified_ok = (m2.Status == GRB.OPTIMAL)

    if verbose:
        if verified_ok:
            print("✅ Existe al menos un óptimo favorable → WCE válido")
        else:
            print("❌ Ningún óptimo favorable → NO es WCE")

    return {
        "verified_ok": verified_ok,
        "optimal_value": z_star,
        "status": m2.Status
    }

In [ ]:
def verify_instance_feasibility_alg3(inst):
    """
    Verifica que X ∩ D ≠ ∅ usando D_mat, d_vec directamente
    """
    n = len(inst["c_hat"])

    m = gp.Model()
    m.Params.OutputFlag = 0

    x = [
        m.addVar(
            lb=inst["bounds"][i][0],
            ub=inst["bounds"][i][1],
            vtype=GRB.CONTINUOUS
        )
        for i in range(n)
    ]

    # A x >= b
    for i in range(len(inst["b"])):
        m.addConstr(
            gp.quicksum(inst["A"][i, j] * x[j] for j in range(n)) >= inst["b"][i]
        )

    # D_mat x <= d_vec
    for i in range(len(inst["d_vec"])):
        m.addConstr(
            gp.quicksum(inst["D_mat"][i, j] * x[j] for j in range(n)) <= inst["d_vec"][i]
        )

    m.setObjective(0, GRB.MINIMIZE)
    m.optimize()

    if m.status == GRB.OPTIMAL:
        return True, np.array([x[i].X for i in range(n)])
    return False, None


In [ ]:
# ---------------------------------------
# Stats + ajuste Normal + test (opcional)
# ---------------------------------------
def summarize(df):
    n_total = len(df)
    # Modificado para considerar varios estados de 'OPTIMAL'
    n_opt = int((df["status"].isin(["OPTIMAL", "OPTIMAL_WCE"])).sum())
    n_tl = int((df["status"] == "TIME_LIMIT").sum())
    p_tl = n_tl / n_total if n_total else float("nan")

    finished = df[~df["status"].isin(["TIME_LIMIT", "ALGORITHM_ERROR", "GENERATION_ERROR", "INSTANCE_INFEASIBLE"])]["runtime_sec"].to_numpy()
    mean_finished = float(np.mean(finished)) if len(finished) else float("nan")
    std_finished = float(np.std(finished, ddof=1)) if len(finished) >= 2 else float("nan")

    # Asegurar que 'time_limit_sec' se trata como un escalar
    tl = float(df["time_limit_sec"].iloc[0]) if not df.empty and "time_limit_sec" in df.columns else np.nan

    censored = df["runtime_sec"].to_numpy().copy()
    # Censorizar solo los que realmente llegaron al límite de tiempo
    censored[df["status"].to_numpy() == "TIME_LIMIT"] = tl
    mean_cens = float(np.mean(censored)) if len(censored) else np.nan
    std_cens = float(np.std(censored, ddof=1)) if len(censored) >= 2 else np.nan

    return pd.DataFrame([{
        "count_total": n_total,
        "count_optimal": n_opt,
        "count_time_limit": n_tl,
        "pct_time_limit": p_tl,
        "mean_finished_sec": mean_finished,
        "std_finished_sec": std_finished,
        "mean_censored_sec": mean_cens,
        "std_censored_sec": std_cens
    }])

# ---------------------------------------
# Gráficos de tiempo y L1 + tiempo por n
# ---------------------------------------
def plot_runtime(df):
    import matplotlib.pyplot as plt

    # Histograma de tiempos
    if not df.empty:
        plt.figure()
        plt.hist(df["runtime_sec"], bins=20, color='skyblue', edgecolor='black')
        plt.xlabel("Tiempo (s)")
        plt.ylabel("Frecuencia")
        plt.title("Distribución de tiempos de ejecución")
        plt.grid(True)
        plt.show()
    else:
        print("DataFrame vacío, no se puede graficar el histograma de tiempos.")

    # Histograma de distancias L1
    # Convertir a numérico y eliminar NaNs para asegurar compatibilidad con np.isfinite
    dist = pd.to_numeric(df["dist_L1"], errors='coerce').dropna()

    if len(dist) > 0:
        plt.figure()
        plt.hist(dist, bins=20, edgecolor='black')
        plt.xlabel("Distancia L1")
        plt.ylabel("Frecuencia")
        plt.title("Distribución de dist_L1 (valores finitos)")
        plt.grid(True)
        plt.show()
    else:
        print("No hay valores finitos de dist_L1 para graficar.")

    # Promedio y desviación por número de variables n
    if not df.empty:
        grouped = df.groupby("n")["runtime_sec"].agg(["mean", "std"]).reset_index()

        plt.figure()
        plt.errorbar(
            grouped["n"], grouped["mean"], yerr=grouped["std"],
            fmt='o-', ecolor='red', capsize=5, markersize=6, color='blue'
        )
        plt.xlabel("Número de variables n")
        plt.ylabel("Tiempo promedio (s)")
        plt.yscale("log")  # <-- eje y en escala logarítmica
        plt.title("Tiempo promedio de ejecución ± desviación estándar")
        plt.grid(True)
        plt.show()
    else:
        print("DataFrame vacío, no se puede graficar el tiempo promedio por n.")

## Varían tanto a como b

In [ ]:
import numpy as np

def generate_knapsack(
    n=10,
    n_zero=3,
    n_a_movable=None,   # cuántos coeficientes de a se pueden mover
    b_movable=True,     # si b se puede mover o no
    rng=None,
):
    """
    Knapsack general (a y b móviles/fijos)
    - A x >= b
    - a ∈ [1,10]^n
    - c_hat ∈ [1,25]^n
    - subset de a es movible
    - b puede ser fijo o movible
    """

    if rng is None:
        rng = np.random.default_rng()

    # =========================
    # 1. Dominio
    # =========================
    bounds = [(0, 10)] * n
    var_types = ["I"] * n

    # =========================
    # 2. Costos
    # =========================
    c_hat = rng.integers(1, 26, size=n)

    # =========================
    # 3. Coeficientes nominales
    # =========================
    a_lb = np.ones(n, dtype=int)
    a_ub = 10 * np.ones(n, dtype=int)
    a_hat = ((a_lb + a_ub) // 2).astype(int)

    # =========================
    # 4. Variables movibles de a
    # =========================
    if n_a_movable is None:
        n_a_movable = n

    idx = rng.choice(n, size=n_a_movable, replace=False)
    a_movable = np.zeros(n, dtype=bool)
    a_movable[idx] = True

    # =========================
    # 5. Conjunto D
    # =========================
    D_mat = np.zeros((n_zero, n), dtype=int)
    d_vec = np.zeros(n_zero, dtype=int)
    for i in range(n_zero):
        D_mat[i, i] = 1

    # =========================
    # 6. Construir x factible real
    # =========================
    x_feasible = np.zeros(n, dtype=int)
    for j in range(n_zero, n):
        x_feasible[j] = rng.integers(3, 7)

    # =========================
    # 7. Construir b factible
    # =========================
    lhs_lb = int(a_lb @ x_feasible)
    lhs_ub = int(a_ub @ x_feasible)

    if lhs_lb + 1 >= lhs_ub:
        b_hat = lhs_lb
    else:
        b_hat = rng.integers(lhs_lb + 1, lhs_ub)

    if b_movable:
        b_lb = int(np.floor(0.9 * b_hat))
        b_ub = int(np.ceil(1.1 * b_hat))
    else:
        b_lb = b_hat
        b_ub = b_hat

    # =========================
    # 8. Restricción base Ax ≥ b_vec (dummy)
    # =========================
    A = np.ones((1, n), dtype=int)
    b_vec = np.array([0])

    return {
        "A": A,
        "b": b_vec,
        "D_mat": D_mat,
        "d_vec": d_vec,
        "c_hat": c_hat,
        "bounds": bounds,
        "var_types": var_types,

        # nominales
        "a_hat": a_hat,
        "b_hat": b_hat,

        # cajas
        "a_lb": a_lb,
        "a_ub": a_ub,
        "b_lb": b_lb,
        "b_ub": b_ub,

        # movilidad
        "a_movable": a_movable,
        "b_movable": b_movable,

        # debug
        "x_feasible": x_feasible,
        "v_feasible": int(c_hat @ x_feasible),
    }



In [ ]:
def solve_problem_8_partial_a(
    v,
    c_hat,
    a_hat,
    b_hat,
    A,
    b_vec,
    bounds,
    var_types,
    d_star,
    D_mat,
    d_vec,
    a_lb,
    a_ub,
    b_lb,
    b_ub,
    movable_idx,
    fixed_idx,
    problem_type="min",
    tolerance=1e-6,
    max_iters=100000,
):
    """
    Algoritmo 2 con:
    - a parcialmente movible
    - b siempre movible
    """

    n = len(c_hat)
    Y = []
    iteration = 0

    movable_idx = set(movable_idx)
    fixed_idx = set(fixed_idx)
    assert movable_idx.union(fixed_idx) == set(range(n))

    while iteration < max_iters:
        iteration += 1

        # ====================================================
        # MASTER
        # ====================================================
        master = gp.Model("master_partial_a")
        master.Params.OutputFlag = 0

        # ----- x variables -----
        x = []
        for i in range(n):
            if var_types[i] == "C":
                x.append(master.addVar(lb=bounds[i][0], ub=bounds[i][1]))
            elif var_types[i] == "B":
                x.append(master.addVar(vtype=GRB.BINARY))
            else:
                x.append(master.addVar(lb=bounds[i][0],
                                       ub=bounds[i][1],
                                       vtype=GRB.INTEGER))

        # ----- a variables (solo movibles) -----
        a = {}
        for i in movable_idx:
            a[i] = master.addVar(lb=a_lb[i], ub=a_ub[i], vtype=GRB.INTEGER)

        # ----- b variable (siempre movible) -----
        b = master.addVar(lb=b_lb, ub=b_ub, vtype=GRB.INTEGER)

        # ----- L1 auxiliaries -----
        u = {}
        for i in movable_idx:
            u[i] = master.addVar(lb=0)
            master.addConstr(a[i] - a_hat[i] <= u[i])
            master.addConstr(a_hat[i] - a[i] <= u[i])

        u_b = master.addVar(lb=0)
        master.addConstr(b - b_hat <= u_b)
        master.addConstr(b_hat - b <= u_b)

        # ----- objective -----
        master.setObjective(
            gp.quicksum(u[i] for i in movable_idx) + u_b,
            GRB.MINIMIZE
        )

        # ----- cortes acumulados -----
        for y in Y:
            expr = gp.LinExpr()
            for i in range(n):
                if i in movable_idx:
                    expr += a[i] * y[i]
                else:
                    expr += a_hat[i] * y[i]

            if problem_type == "min":
                master.addConstr(expr <= b - 1)
            else:
                master.addConstr(expr >= b + 1)

        # ----- fijar valor objetivo -----
        master.addConstr(
            gp.quicksum(c_hat[i] * x[i] for i in range(n)) == v
        )

        # ----- restricción clave -----
        expr_ax = gp.LinExpr()
        for i in range(n):
            if i in movable_idx:
                expr_ax += a[i] * x[i]
            else:
                expr_ax += a_hat[i] * x[i]

        if problem_type == "min":
            master.addConstr(expr_ax >= b)
        else:
            master.addConstr(expr_ax <= b)

        # ----- x ∈ X -----
        for i in range(len(b_vec)):
            master.addConstr(
                gp.quicksum(A[i, j] * x[j] for j in range(n)) >= b_vec[i]
            )

        # ----- x ∈ D -----
        for i in range(len(d_vec)):
            master.addConstr(
                gp.quicksum(D_mat[i, j] * x[j] for j in range(n)) <= d_vec[i]
            )

        # ----- presupuesto -----
        if np.isfinite(d_star):
            master.addConstr(
                gp.quicksum(u[i] for i in movable_idx) + u_b <= d_star
            )

        master.optimize()
        if master.status != GRB.OPTIMAL:
            return None, None

        # ----- soluciones -----
        a_sol = a_hat.copy()
        for i in movable_idx:
            a_sol[i] = a[i].X
        b_sol = b.X

        # ====================================================
        # SEPARATION: soluciones estrictamente mejores
        # ====================================================
        sep = gp.Model("sep_strict")
        sep.Params.OutputFlag = 0

        y = []
        for i in range(n):
            if var_types[i] == "C":
                y.append(sep.addVar(lb=bounds[i][0], ub=bounds[i][1]))
            elif var_types[i] == "B":
                y.append(sep.addVar(vtype=GRB.BINARY))
            else:
                y.append(sep.addVar(lb=bounds[i][0],
                                   ub=bounds[i][1],
                                   vtype=GRB.INTEGER))

        for i in range(len(b_vec)):
            sep.addConstr(
                gp.quicksum(A[i, j] * y[j] for j in range(n)) >= b_vec[i]
            )

        sep.addConstr(gp.quicksum(c_hat[i] * y[i] for i in range(n)) <= v - 1)
        sep.setObjective(
            gp.quicksum(a_sol[i] * y[i] for i in range(n)),
            GRB.MAXIMIZE
        )
        sep.optimize()

        if sep.status == GRB.OPTIMAL:
            y_sol = np.array([yi.X for yi in y])
            if np.dot(a_sol, y_sol) >= b_sol - tolerance:
                Y.append(y_sol)
                continue

        # ====================================================
        # CHECK WCE (óptimo en D)
        # ====================================================
        sep_in_D = gp.Model("sep_in_D")
        sep_in_D.Params.OutputFlag = 0

        y = []
        for i in range(n):
            if var_types[i] == "C":
                y.append(sep_in_D.addVar(lb=bounds[i][0], ub=bounds[i][1]))
            elif var_types[i] == "B":
                y.append(sep_in_D.addVar(vtype=GRB.BINARY))
            else:
                y.append(sep_in_D.addVar(lb=bounds[i][0],
                                         ub=bounds[i][1],
                                         vtype=GRB.INTEGER))

        for i in range(len(b_vec)):
            sep_in_D.addConstr(
                gp.quicksum(A[i, j] * y[j] for j in range(n)) >= b_vec[i]
            )

        for i in range(len(d_vec)):
            sep_in_D.addConstr(
                gp.quicksum(D_mat[i, j] * y[j] for j in range(n)) <= d_vec[i]
            )

        sep_in_D.addConstr(
            gp.quicksum(c_hat[i] * y[i] for i in range(n)) == v
        )

        sep_in_D.setObjective(0.0, GRB.MINIMIZE)
        sep_in_D.optimize()

        if sep_in_D.status == GRB.OPTIMAL:
            break

        # ----- empate fuera de D -----
        sep_any = gp.Model("sep_any")
        sep_any.Params.OutputFlag = 0
        y = [sep_any.addVar(lb=bounds[i][0],
                            ub=bounds[i][1],
                            vtype=GRB.INTEGER)
             for i in range(n)]

        for i in range(len(b_vec)):
            sep_any.addConstr(
                gp.quicksum(A[i, j] * y[j] for j in range(n)) >= b_vec[i]
            )

        sep_any.addConstr(
            gp.quicksum(c_hat[i] * y[i] for i in range(n)) == v
        )

        sep_any.setObjective(
            gp.quicksum(a_sol[i] * y[i] for i in range(n)),
            GRB.MAXIMIZE
        )
        sep_any.optimize()

        if sep_any.status == GRB.OPTIMAL:
            Y.append(np.array([yi.X for yi in y]))
            continue

        break

    return a_sol, b_sol


In [ ]:
import numpy as np
def compute_c_bounds(c_hat, A, b_vec, D_mat, d_vec, a_ub, b_lb, bounds, var_types, time_limit=5):
    """
    Calcula cotas c_min y c_max para el problema con manejo robusto de infeasibilidad.

    ✅ CORRECCIÓN: Si el problema con restricción a_max'x >= b_min es infeasible,
    intenta sin esa restricción y usa cotas muy conservadoras.
    """
    n = len(c_hat)
    a_max = a_ub
    b_min = b_lb

    def build_model(sense, include_ab_constraint=True):
        m = gp.Model()
        m.Params.OutputFlag = 0
        m.Params.TimeLimit = time_limit
        m.Params.MIPFocus = 1  # Enfocarse en encontrar soluciones factibles

        x = []
        for i in range(n):
            if var_types[i] == 'B':
                x.append(m.addVar(vtype=GRB.BINARY))
            elif var_types[i] == 'I':
                x.append(m.addVar(lb=bounds[i][0], ub=bounds[i][1], vtype=GRB.INTEGER))
            else:
                x.append(m.addVar(lb=bounds[i][0], ub=bounds[i][1]))

        # Restricciones A x >= b_vec
        for i in range(A.shape[0]):
            m.addConstr(gp.quicksum(A[i,j]*x[j] for j in range(n)) >= b_vec[i])

        # Restricciones D (conjunto favorable)
        for i in range(D_mat.shape[0]):
            m.addConstr(gp.quicksum(D_mat[i,j]*x[j] for j in range(n)) <= d_vec[i])

        # ✅ Restricción opcional a_max'x >= b_min
        if include_ab_constraint:
            m.addConstr(gp.quicksum(a_max[i]*x[i] for i in range(n)) >= b_min)

        m.setObjective(gp.quicksum(c_hat[i]*x[i] for i in range(n)), sense)
        return m

    # ========================================
    # Intento 1: Con restricción a_max'x >= b_min
    # ========================================
    try:
        m_min = build_model(GRB.MINIMIZE, include_ab_constraint=True)
        m_min.optimize()

        if m_min.status == GRB.OPTIMAL:
            c_min = m_min.ObjVal
        elif m_min.status == GRB.TIME_LIMIT and m_min.SolCount > 0:
            c_min = m_min.ObjVal
        else:
            raise RuntimeError("Infeasible with ab_constraint")

        m_max = build_model(GRB.MAXIMIZE, include_ab_constraint=True)
        m_max.optimize()

        if m_max.status == GRB.OPTIMAL:
            c_max = m_max.ObjVal
        elif m_max.status == GRB.TIME_LIMIT and m_max.SolCount > 0:
            c_max = m_max.ObjVal
        else:
            raise RuntimeError("Infeasible with ab_constraint")

        return int(np.floor(c_min)), int(np.ceil(c_max))

    except Exception as e:
        # Print the status when an error occurs
        status_min = m_min.status if 'm_min' in locals() else 'N/A'
        status_max = m_max.status if 'm_max' in locals() else 'N/A'
        print(f"compute_c_bounds (Exception attempt 1): Status m_min={status_min}, m_max={status_max}, Error: {e}")
        pass  # Continuar al intento 2

    # ========================================
    # Intento 2: SIN restricción a_max'x >= b_min
    # ========================================
    try:
        m_min = build_model(GRB.MINIMIZE, include_ab_constraint=False)
        m_min.optimize()

        if m_min.status not in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
            raise RuntimeError("compute_c_bounds: Problema base infeasible")

        c_min = m_min.ObjVal if m_min.status == GRB.OPTIMAL else m_min.ObjBound

        m_max = build_model(GRB.MAXIMIZE, include_ab_constraint=False)
        m_max.optimize()

        if m_max.status not in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
            raise RuntimeError("compute_c_bounds: Problema base infeasible")

        c_max = m_max.ObjVal if m_max.status == GRB.OPTIMAL else m_max.ObjBound

        # ✅ Expandir cotas conservadoramente
        margin = max(10, int(0.2 * abs(c_max - c_min)))
        return int(np.floor(c_min - margin)), int(np.ceil(c_max + margin))

    except Exception as e:
        status_min = m_min.status if 'm_min' in locals() else 'N/A'
        status_max = m_max.status if 'm_max' in locals() else 'N/A'
        raise RuntimeError(f"compute_c_bounds: Infeasible incluso sin restricción ab: {e}")


def compute_lower_bound_lemma4_L1(
    c_hat,
    v_bar,
    A,
    b_vec,
    bounds,
    var_types,
    a_hat,
    b_hat,
    a_lb,
    a_ub,
    b_lb,
    b_ub,
    time_limit,
    max_cuts = 100
):

    t0 = time.perf_counter()
    n = len(c_hat)
    p = len(a_hat)

    m = gp.Model()
    m.Params.OutputFlag = 0
    m.Params.TimeLimit = time_limit
    m.Params.MIPFocus = 1  # Enfocarse en encontrar soluciones factibles

    a = m.addVars(p, lb=a_lb, ub=a_ub, vtype=GRB.INTEGER)
    b = m.addVar(lb=b_lb, ub=b_ub, vtype=GRB.INTEGER)

    u = m.addVars(p, lb=0)
    v_aux = m.addVar(lb=0)

    for i in range(p):
        m.addConstr(a[i] - a_hat[i] <= u[i])
        m.addConstr(a_hat[i] - a[i] <= u[i])

    m.addConstr(b - b_hat <= v_aux)
    m.addConstr(b_hat - b <= v_aux)

    for i in range(p):
        m.addConstr(a[i] >= a_lb[i])
        m.addConstr(a[i] <= a_ub[i])

    m.addConstr(b >= b_lb)
    m.addConstr(b <= b_ub)


    m.setObjective(gp.quicksum(u[i] for i in range(p)) + v_aux, GRB.MINIMIZE)

    cuts = 0
    while cuts < max_cuts:
        if time.perf_counter() - t0 > time_limit:
            return np.inf

        m.optimize()
        if m.status != GRB.OPTIMAL:
            return np.inf

        a_val = np.array([a[i].X for i in range(p)])
        b_val = b.X

        sep = gp.Model()
        sep.Params.OutputFlag = 0
        sep.Params.TimeLimit = time_limit
        sep.Params.MIPFocus = 1  # Enfocarse en encontrar soluciones factibles

        y = []
        for i in range(n):
            if var_types[i] == 'B':
                y.append(sep.addVar(vtype=GRB.BINARY))
            elif var_types[i] == 'I':
                y.append(sep.addVar(lb=bounds[i][0], ub=bounds[i][1], vtype=GRB.INTEGER))
            else:
                y.append(sep.addVar(lb=bounds[i][0], ub=bounds[i][1]))

        for i in range(A.shape[0]):
            sep.addConstr(gp.quicksum(A[i,j]*y[j] for j in range(n)) >= b_vec[i])

        sep.addConstr(gp.quicksum(c_hat[i]*y[i] for i in range(n)) <= v_bar - 1)
        sep.setObjective(gp.quicksum(a_val[i]*y[i] for i in range(n)), GRB.MAXIMIZE)
        sep.optimize()

        if sep.status != GRB.OPTIMAL or sep.ObjVal <= b_val - 1 + 1e-6:
            break

        y_star = np.array([y[i].X for i in range(n)])
        m.addConstr(gp.quicksum(a[i]*y_star[i] for i in range(n)) <= b - 1)
        cuts += 1

    return m.ObjVal


def algorithm3_partial_a(
    c_hat,
    a_hat,
    b_hat,
    A,
    b_vec,
    a_lb,
    a_ub,
    b_lb,
    b_ub,
    bounds,
    var_types,
    D_mat,
    d_vec,
    movable_idx,
    fixed_idx,
    sense="min",
    time_limit_seconds=120,
):
    """
    Algorithm 3 – Optimal Weak CE
    with partially movable a and fully movable b
    """

    t0 = time.perf_counter()

    a_star_best = None
    b_star_best = None
    d_star = np.inf
    stopped_by_lemma4 = False

    # ====================================================
    # Cotas de v
    # ====================================================
    c_min, c_max = compute_c_bounds(
        c_hat,
        A,
        b_vec,
        D_mat,
        d_vec,
        a_ub,
        b_lb,
        bounds,
        var_types,
        time_limit=5,
    )

    # ====================================================
    # Loop principal en v
    # ====================================================
    for v in range(c_min, c_max + 1):

        # ⏱️ Time limit global
        if time.perf_counter() - t0 >= time_limit_seconds:
            print("⏱️ Time limit alcanzado en Algorithm 3 (partial a)")
            break

        # ====================================================
        # Lemma 4 lower bound (MISMO concepto)
        # Nota: sigue usando ||a-â||₁ + |b-b̂|
        # aunque a sea parcialmente movible
        # ====================================================
        lb = compute_lower_bound_lemma4_L1(
            c_hat,
            v,
            A,
            b_vec,
            bounds,
            var_types,
            a_hat,
            b_hat,
            a_lb,
            a_ub,
            b_lb,
            b_ub,
            time_limit=30,
            max_cuts=500,
        )

        if a_star_best is not None and lb >= d_star:
            print("Stop early, Lemma 4 lower bound ≥ d_star")
            stopped_by_lemma4 = True
            break

        # ====================================================
        # Resolver Problem (8) parcial
        # ====================================================
        result = solve_problem_8_partial_a(
            v=v,
            c_hat=c_hat,
            a_hat=a_hat,
            b_hat=b_hat,
            A=A,
            b_vec=b_vec,
            bounds=bounds,
            var_types=var_types,
            d_star=d_star,
            D_mat=D_mat,
            d_vec=d_vec,
            a_lb=a_lb,
            a_ub=a_ub,
            b_lb=b_lb,
            b_ub=b_ub,
            movable_idx=movable_idx,
            fixed_idx=fixed_idx,
            problem_type=sense,
        )

        if result is None or result[0] is None or result[1] is None:
            continue

        a_star_v, b_star_v = result

        # ====================================================
        # Distancia L1 (SOLO en índices movibles + b)
        # ====================================================
        dist_v = (
            sum(abs(a_star_v[i] - a_hat[i]) for i in movable_idx)
            + abs(b_star_v - b_hat)
        )

        print(f"v={v}: Distancia parcial obtenida = {dist_v}")

        # ====================================================
        # Mejorar incumbent
        # ====================================================
        if dist_v < d_star:
            a_star_best = a_star_v.copy()
            b_star_best = b_star_v
            d_star = dist_v

    return {
        "a_star_best": a_star_best,
        "b_star_best": b_star_best,
        "d_star": d_star,
        "c_min": c_min,
        "c_max": c_max,
        "iters": (v - c_min + 1) if "v" in locals() else 0,
        "stopped_by_lemma4": stopped_by_lemma4,
    }


In [ ]:
def run_trial_knapsack(n, n_zero, seed, n_a_movable ,alg_time_limit=120):
    rng = np.random.default_rng(seed)

    # =========================
    # 1. Generar instancia
    # =========================
    try:
        inst = generate_knapsack(n=n, n_zero=n_zero, n_a_movable = n_a_movable, rng=rng)
    except Exception as e:
        return {
            "seed": seed,
            "n": n,
            "status": "GENERATION_ERROR",
            "runtime_sec": 0.0,
            "iters": 0,
            "verified_ok": False,
            "error": str(e)[:80],
        }

    # =========================
    # 2. Verificar X ∩ D ≠ ∅
    # =========================
    is_feas, _ = verify_instance_feasibility_alg3(inst)
    if not is_feas:
        return {
            "seed": seed,
            "n": n,
            "status": "INSTANCE_INFEASIBLE",
            "runtime_sec": 0.0,
            "iters": 0,
            "verified_ok": False,
        }

    # =========================
    # 3. Índices movibles / fijos de a
    # =========================
    movable_idx = np.where(inst["a_movable"])[0].tolist()
    fixed_idx = np.where(~inst["a_movable"])[0].tolist()

    # =========================
    # 4. Ejecutar Algorithm 3 (parcial a)
    # =========================
    t0 = time.perf_counter()
    try:
        res = algorithm3_partial_a(
            c_hat=inst["c_hat"],
            a_hat=inst["a_hat"],
            b_hat=inst["b_hat"],
            A=inst["A"],
            b_vec=inst["b"],
            a_lb=inst["a_lb"],
            a_ub=inst["a_ub"],
            b_lb=inst["b_lb"],
            b_ub=inst["b_ub"],
            bounds=inst["bounds"],
            var_types=inst["var_types"],
            D_mat=inst["D_mat"],
            d_vec=inst["d_vec"],
            movable_idx=movable_idx,
            fixed_idx=fixed_idx,
            sense="min",
            time_limit_seconds=alg_time_limit,
        )
    except Exception as e:
        return {
            "seed": seed,
            "n": n,
            "status": "ALGORITHM_ERROR",
            "runtime_sec": time.perf_counter() - t0,
            "iters": 0,
            "verified_ok": False,
            "error": str(e)[:80],
        }

    runtime = time.perf_counter() - t0

    # =========================
    # 5. Interpretar resultado
    # =========================
    a_star = res.get("a_star_best")
    b_star = res.get("b_star_best")

    if a_star is not None and b_star is not None:
        dist_L1 = (
            sum(abs(a_star[i] - inst["a_hat"][i]) for i in movable_idx)
            + abs(b_star - inst["b_hat"])
        )
        algo_status = "OPTIMAL"
    elif res.get("stopped_by_lemma4", False):
        dist_L1 = None
        algo_status = "NO_IMPROVEMENT"
    else:
        return {
            "seed": seed,
            "n": n,
            "runtime_sec": runtime,
            "time_limit_sec": alg_time_limit,
            "status": "INFEASIBLE",
            "iters": res.get("iters", 0),
            "d_star": None,
            "dist_L1": None,
            "verified_ok": False,
        }

    # =========================
    # 6. Verificación WCE
    # =========================
    ver = verify_algorithm3_solution(
        a_star=a_star,
        b_star=b_star,
        c_hat=inst["c_hat"],
        A=inst["A"],
        b_vec=inst["b"],
        D_mat=inst["D_mat"],
        d_vec=inst["d_vec"],
        bounds=inst["bounds"],
        var_types=inst["var_types"],
        tol=1e-6,
        verbose=False,
    )

    verified_ok = ver["verified_ok"]

    final_status = (
        "OPTIMAL_WCE" if verified_ok else "OPTIMAL_NOT_WCE"
        if algo_status == "OPTIMAL"
        else algo_status
    )

    # =========================
    # 7. Salida final
    # =========================
    return {
        "seed": seed,
        "n": n,
        "runtime_sec": runtime,
        "time_limit_sec": alg_time_limit,
        "status": final_status,
        "iters": res.get("iters", 0),
        "d_star": None if res.get("d_star", np.inf) == np.inf else res.get("d_star"),
        "dist_L1": dist_L1,
        "verified_ok": verified_ok,
        "n_a_movable": len(movable_idx),
    }

def benchmark_trial_knapsack(ns, n_zero, n_a_movable, trials_per_n=10, base_seed=200):
    """Benchmark para Algorithm 3 con a parcialmente movible"""
    rows = []
    seed = base_seed

    total_trials = len(ns) * trials_per_n
    completed = 0

    for n in ns:
        for _ in range(trials_per_n):
            rows.append(run_trial_knapsack(n, n_zero, seed, n_a_movable))
            seed += 1
            completed += 1

            progress = completed / total_trials
            bar_length = 40
            filled_length = int(bar_length * progress)
            bar = "█" * filled_length + "-" * (bar_length - filled_length)
            print(
                f"\rProgreso: |{bar}| {progress*100:.1f}% ({completed}/{total_trials})",
                end="",
            )

    print()
    return pd.DataFrame(rows)


In [ ]:
ns = [6]
df = benchmark_trial_knapsack (ns, n_zero = 2, n_a_movable = 5, trials_per_n=1)

print(df.groupby(["n", "status", "verified_ok"]).size())
print(df.describe())
summary = summarize(df)
print(summary)
df_opt = df[df["status"] == "OPTIMAL_WCE"].reset_index(drop=True)
plot_runtime(df_opt)

In [ ]:
def benchmark_grid_partial_mobility(
    n,
    trials_per_cell=5,
    base_seed=200,
):
    """
    Meta-benchmark que usa benchmark_trial_knapsack internamente.

    Devuelve DataFrame agregado con:
      - n_zero
      - n_a_movable
      - tiempo promedio
      - porcentaje factible (OPTIMAL_WCE)
    """

    rows = []
    seed = base_seed

    for n_zero in range(1, 5):
        for n_a_movable in range(1, 5):

            print(f"\n▶ n_zero={n_zero}, n_a_movable={n_a_movable}")

            df = benchmark_trial_knapsack(
                ns=[n],
                n_zero=n_zero,
                n_a_movable=n_a_movable,
                trials_per_n=trials_per_cell,
                base_seed=seed,
            )

            seed += trials_per_cell

            df_ok = df[df["status"] == "OPTIMAL_WCE"]
            if len(df_ok) > 0:
                avg_runtime = df_ok["runtime_sec"].mean()
            else:
                avg_runtime = np.nan


            feasible_rate = np.mean(
                df["status"] == "OPTIMAL_WCE"
            )

            rows.append({
                "n": n,
                "n_zero": n_zero,
                "n_a_movable": n_a_movable,
                "avg_runtime": avg_runtime,
                "feasible_rate": feasible_rate,
            })

    return pd.DataFrame(rows)


In [ ]:
def build_confusion_matrices(df):
    """
    Construye matrices 4x4:
      filas  -> n_zero = 1..4
      columnas -> n_a_movable = 1..4
    """

    mat_time = np.full((4, 4), np.nan)
    mat_feas = np.full((4, 4), np.nan)

    for _, row in df.iterrows():
        i = row["n_zero"] - 1
        j = row["n_a_movable"] - 1
        mat_time[i, j] = row["avg_runtime"]
        mat_feas[i, j] = row["feasible_rate"]

    return mat_time, mat_feas


In [ ]:
import matplotlib.pyplot as plt

def plot_confusion_matrix_time_feas(mat_time, mat_feas, n):
    fig, ax = plt.subplots(figsize=(8, 6))

    im = ax.imshow(mat_time, origin="lower")

    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels([1, 2, 3, 4])
    ax.set_yticklabels([1, 2, 3, 4])

    ax.set_xlabel("Número de coeficientes de a movibles")
    ax.set_ylabel("n_zero")
    ax.set_title(f"Benchmark Algorithm 3 – movilidad parcial (n = {n})")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Tiempo promedio (s)")

    # Texto dentro de cada celda
    for i in range(4):
        for j in range(4):
            if np.isnan(mat_time[i, j]):
                continue
            txt = (
                f"{mat_time[i, j]:.2f}s\n"
                f"{100 * mat_feas[i, j]:.0f}%"
            )
            ax.text(j, i, txt, ha="center", va="center", color="white", fontsize=10)

    plt.tight_layout()
    plt.show()


In [ ]:
ns = [8,9,10,11,12]

def build_confusion_matrices(df):
    """
    Construye matrices 4x4:
      filas  -> n_zero = 1..4
      columnas -> n_a_movable = 1..4
    """

    mat_time = np.full((4, 4), np.nan)
    mat_feas = np.full((4, 4), np.nan)

    for _, row in df.iterrows():
        # Explicitly cast to int to prevent IndexError if they become float
        i = int(row["n_zero"]) - 1
        j = int(row["n_a_movable"]) - 1
        mat_time[i, j] = row["avg_runtime"]
        mat_feas[i, j] = row["feasible_rate"]

    return mat_time, mat_feas


for n in ns:
    df_grid = benchmark_grid_partial_mobility(
        n=n,
        trials_per_cell=20,
    )

    mat_time, mat_feas = build_confusion_matrices(df_grid)

    plot_confusion_matrix_time_feas(mat_time, mat_feas, n)


## Variar solo el valor de b con una variable

In [ ]:
import numpy as np


def generate_knapsack_b_only_robust(
    n=10,
    n_zero=2,
    m_A=4,
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng()

    # ==================================================
    # 1. Índices
    # ==================================================
    idx_bad = np.arange(n_zero)          # prohibidas
    idx_shared = np.arange(n_zero, n_zero + 2)
    idx_good = np.arange(n_zero + 2, n)

    # ==================================================
    # 2. Construir x_bad y x_good
    # ==================================================
    x_bad = np.zeros(n, dtype=int)
    x_good = np.zeros(n, dtype=int)

    # x_bad usa prohibidas (baratas pero ineficientes)
    x_bad[idx_bad] = rng.integers(3, 6, size=len(idx_bad))
    x_good[idx_bad] = 0

    # compartidas iguales
    x_shared = rng.integers(2, 4, size=len(idx_shared))
    x_bad[idx_shared] = x_shared
    x_good[idx_shared] = x_shared

    # x_good usa permitidas (caras pero eficientes)
    x_bad[idx_good] = rng.integers(0, 2, size=len(idx_good))
    x_good[idx_good] = rng.integers(4, 7, size=len(idx_good))

    # ==================================================
    # 3. Conjunto D
    # ==================================================
    D_mat = np.zeros((len(idx_bad), n), dtype=int)
    d_vec = np.zeros(len(idx_bad), dtype=int)
    for i, j in enumerate(idx_bad):
        D_mat[i, j] = 1  # x_j <= 0

    # ==================================================
    # 4. a: prohibidas ineficientes
    # ==================================================
    a = np.zeros(n, dtype=int)
    a[idx_bad] = rng.integers(1, 2, size=len(idx_bad))     # poca contribución
    a[idx_shared] = rng.integers(3, 5, size=len(idx_shared))
    a[idx_good] = rng.integers(6, 9, size=len(idx_good))   # muy eficientes

    b_hat = int(a @ x_bad)
    b_switch = int(a @ x_good)
    assert b_switch > b_hat

    # ==================================================
    # 5. c: prohibidas baratas
    # ==================================================
    c = np.zeros(n, dtype=int)
    c[idx_bad] = rng.integers(1, 3, size=len(idx_bad))     # baratas
    c[idx_shared] = rng.integers(3, 5, size=len(idx_shared))
    c[idx_good] = rng.integers(6, 9, size=len(idx_good))   # caras

    assert c @ x_bad < c @ x_good  # clave

    # ==================================================
    # 6. Restricciones Ax >= b_vec
    # ==================================================
    A = []
    b_vec = []

    for _ in range(m_A):
        r = rng.integers(0, 4, size=n)
        if np.all(r == 0):
            r[rng.integers(0, n)] = 1

        b_i = min(r @ x_bad, r @ x_good)
        A.append(r)
        b_vec.append(b_i)

    A = np.array(A, dtype=int)
    b_vec = np.array(b_vec, dtype=int)

    # ==================================================
    # 7. Intervalo de b
    # ==================================================
    b_lb = 0
    b_ub = 1000

    # ==================================================
    # 8. Checks finales
    # ==================================================
    assert np.all(A @ x_bad >= b_vec)
    assert np.all(A @ x_good >= b_vec)
    assert np.any(x_bad[idx_bad] > 0)
    assert np.all(x_good[idx_bad] == 0)

    return {
        "A": A,
        "b": b_vec,
        "c_hat": c,
        "a_hat": a,
        "b_hat": b_hat,
        "b_lb": b_lb,
        "b_ub": b_ub,
        "D_mat": D_mat,
        "d_vec": d_vec,
        "x_bad": x_bad,
        "x_good": x_good,
        "b_switch": b_switch,
    }




In [ ]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB


def algorithm3_b_remark1(
    c,
    A,
    b_vec,
    a,
    b_central,
    b_lb,
    b_ub,
    D_mat=None,
    d_vec=None,
    time_limit_seconds=120,
):
    """
    Itera sobre b en orden de distancia a b_central y resuelve:
    1) min c^T x s.t. Ax>=b_vec, a^T x >= b
    2) min c^T x s.t. Ax>=b_vec, a^T x >= b, D_mat*x <= d_vec
    Variables enteras entre 0 y 10.
    """

    b_candidates = list(range(b_lb, b_ub + 1))
    b_candidates.sort(key=lambda b: abs(b - b_central))

    n_vars = len(c)
    A = np.atleast_2d(A)
    b_vec = np.atleast_1d(b_vec)

    iters = 0
    for b_val in b_candidates:
        iters += 1

        c_val = -1
        c_val_D = -1

        # =============================
        # Problema original
        # =============================
        m1 = gp.Model()
        m1.Params.OutputFlag = 0
        m1.Params.TimeLimit = time_limit_seconds

        x1 = [m1.addVar(vtype=GRB.INTEGER, lb=0, ub=10) for _ in range(n_vars)]

        for i in range(A.shape[0]):
            m1.addConstr(
                gp.quicksum(A[i, j] * x1[j] for j in range(n_vars)) >= b_vec[i]
            )

        m1.addConstr(gp.quicksum(a[j] * x1[j] for j in range(n_vars)) >= b_val)

        m1.setObjective(
            gp.quicksum(c[j] * x1[j] for j in range(n_vars)), GRB.MINIMIZE
        )
        m1.optimize()

        if m1.status == GRB.OPTIMAL:
            c_val = m1.ObjVal
            x_opt = np.array([v.X for v in x1])
        elif m1.status == GRB.TIME_LIMIT:
            print("Problema original: TIME LIMIT")
            continue


        # =============================
        # Problema con D
        # =============================
        if D_mat is not None and d_vec is not None:
            m2 = gp.Model()
            m2.Params.OutputFlag = 0
            m2.Params.TimeLimit = time_limit_seconds

            x2 = [m2.addVar(vtype=GRB.INTEGER, lb=0, ub=10) for _ in range(n_vars)]

            for i in range(A.shape[0]):
                m2.addConstr(
                    gp.quicksum(A[i, j] * x2[j] for j in range(n_vars)) >= b_vec[i]
                )

            m2.addConstr(gp.quicksum(a[j] * x2[j] for j in range(n_vars)) >= b_val)

            for i in range(D_mat.shape[0]):
                m2.addConstr(
                    gp.quicksum(D_mat[i, j] * x2[j] for j in range(n_vars))
                    <= d_vec[i]
                )

            m2.setObjective(
                gp.quicksum(c[j] * x2[j] for j in range(n_vars)), GRB.MINIMIZE
            )
            m2.optimize()

            if m2.status == GRB.OPTIMAL:
                c_val_D = m2.ObjVal
                x_opt_D = np.array([v.X for v in x2])
            elif m2.status == GRB.TIME_LIMIT:
                print("Problema con D_mat: TIME LIMIT")
                continue

        # =============================
        # Chequeo weak-CE
        # =============================
        if c_val != -1 and c_val_D != -1 and abs(c_val_D - c_val) <= 1e-4:
            print("Coincide === TRUE")
            return {
                "a_star_best": a.copy(),
                "b_star_best": b_val,
                "d_star": abs(b_val - b_central),
                "FACTIBLE": True,
                "iters": iters,
            }

        if c_val == -1 and c_val_D == -1:
            print ("Problema INFACTIBLE")
            return {
                "a_star_best": a.copy(),
                "b_star_best": b_val,
                "d_star": abs(b_val - b_central),
                "FACTIBLE": False,
                "iters": iters,
            }

    return {"FACTIBLE": False, "iters": iters}



n = 3
c = np.array([1,1,1])
A = np.array([[1,1,1]])
b_vec = np.array([5])
a_hat = np.array([1,1,1])

b_hat = 8
b_lb = 0
b_ub = 10

D_mat = np.array([[1,0,0]])
d_vec = np.array([0])

# =========================================
# Ejecutar
# =========================================
algorithm3_b_remark1(c, A, b_vec, a_hat, b_central=b_hat, b_lb=b_lb, b_ub=b_ub,
                        D_mat=D_mat, d_vec=d_vec)



In [ ]:
inst =  generate_knapsack_b_only_robust(n=10, n_zero=2)
print (inst)
A = inst["A"]
print (f"A: {A}")
b_vec = inst["b"] # Corrected: use inst["b"] instead of inst["b_hat"]
c = inst["c_hat"]
a_hat = inst ["a_hat"]

b_hat = inst ["b_hat"]
b_lb = inst ["b_lb"]
b_ub = inst ["b_ub"]

D_mat = inst ["D_mat"]
d_vec = inst ["d_vec"]

print("A:", type(A), getattr(A, "shape", None))
print("b_vec:", type(b_vec))


res = algorithm3_b_remark1(c, A, b_vec, a_hat, b_central=b_hat, b_lb=b_lb, b_ub=b_ub,
                        D_mat=D_mat, d_vec=d_vec)
print (res)

In [ ]:
import time
import pandas as pd
import numpy as np

def run_trial_knapsack_b_enumeration(n, n_zero, seed, alg_time_limit=120):
    rng = np.random.default_rng(seed)

    # =========================
    # 1. Generar instancia
    # =========================
    try:
        inst = generate_knapsack_b_only_robust(
            n=n,
            n_zero=n_zero,
            rng=rng,
             m_A=1
        )
    except Exception as e:
        return {
            "seed": seed,
            "n": n,
            "runtime_sec": 0.0,
            "time_limit_sec": alg_time_limit,
            "status": "GENERATION_ERROR",
            "iters": 0,
            "d_star": None,
            "dist_L1": None,
            "verified_ok": False,
            "error": str(e)[:200],
        }

    # =========================
    # 2. Ejecutar Algorithm 3-b (Remark 1)
    # =========================
    t0 = time.perf_counter()
    try:
        res = algorithm3_b_remark1(
            c=inst["c_hat"],
            a=inst["a_hat"],
            b_central=inst["b_hat"],
            A=inst["A"],
            b_vec=inst["b"],
            b_lb=inst["b_lb"],
            b_ub=inst["b_ub"],
            D_mat=inst["D_mat"],
            d_vec=inst["d_vec"],
            time_limit_seconds=alg_time_limit,
        )
    except Exception as e:
        runtime = time.perf_counter() - t0
        return {
            "seed": seed,
            "n": n,
            "runtime_sec": runtime,
            "time_limit_sec": alg_time_limit,
            "status": "ALGORITHM_ERROR",
            "iters": 0,
            "d_star": None,
            "dist_L1": None,
            "verified_ok": False,
            "error": str(e)[:200],
        }

    runtime = time.perf_counter() - t0

    # =========================
    # 3. Interpretar resultado
    # =========================
    b_star = res.get("b_star_best")
    iters = res.get("iters", 0)
    feasible = res.get("FACTIBLE", False)
    d_star = res.get("d_star")

    if b_star is not None:
        dist_L1 = abs(b_star - inst["b_hat"])
    else:
        dist_L1 = None

    verified_ok = feasible and (b_star is not None)

    final_status = "OPTIMAL_WCE" if verified_ok else "INFEASIBLE"

    # =========================
    # 4. Salida final
    # =========================
    return {
        "seed": seed,
        "n": n,
        "runtime_sec": runtime,
        "time_limit_sec": alg_time_limit,
        "status": final_status,
        "iters": iters,
        "d_star": d_star,
        "dist_L1": dist_L1,
        "verified_ok": verified_ok,
        "error": "",
    }



def benchmark_trial_knapsack_b_enumeration(ns, n_zero, trials_per_n=10, base_seed=200):
    rows = []
    seed = base_seed
    total_trials = len(ns) * trials_per_n
    completed = 0

    print("\n--- Running Enumeration-based Benchmark ---")
    for n in ns:
        for _ in range(trials_per_n):
            rows.append(run_trial_knapsack_b_enumeration(n, n_zero, seed))
            seed += 1
            completed += 1

            progress = completed / total_trials
            bar_length = 40
            filled_length = int(bar_length * progress)
            bar = "█" * filled_length + "-" * (bar_length - filled_length)
            print(
                f"\rProgreso: |{bar}| {progress*100:.1f}% ({completed}/{total_trials})",
                end=""
            )

    print()
    return pd.DataFrame(rows)


In [ ]:
# ---------------------------------------
# Stats + ajuste Normal + test (opcional)
# ---------------------------------------
def summarize(df):
    n_total = len(df)
    # Modificado para considerar varios estados de 'OPTIMAL'
    n_opt = int((df["status"].isin(["OPTIMAL", "OPTIMAL_WCE"]).sum()))
    n_tl = int((df["status"] == "TIME_LIMIT").sum())
    p_tl = n_tl / n_total if n_total else float("nan")

    finished = df[~df["status"].isin(["TIME_LIMIT", "ALGORITHM_ERROR", "GENERATION_ERROR", "INSTANCE_INFEASIBLE"])]["runtime_sec"].to_numpy()
    mean_finished = float(np.mean(finished)) if len(finished) else float("nan")
    std_finished = float(np.std(finished, ddof=1)) if len(finished) >= 2 else float("nan")

    # Asegurar que 'time_limit_sec' se trata como un escalar
    tl = float(df["time_limit_sec"].iloc[0]) if not df.empty and "time_limit_sec" in df.columns else np.nan

    censored = df["runtime_sec"].to_numpy().copy()
    # Censorizar solo los que realmente llegaron al límite de tiempo
    censored[df["status"].to_numpy() == "TIME_LIMIT"] = tl
    mean_cens = float(np.mean(censored)) if len(censored) else np.nan
    std_cens = float(np.std(censored, ddof=1)) if len(censored) >= 2 else float("nan")

    return pd.DataFrame([{
        "count_total": n_total,
        "count_optimal": n_opt,
        "count_time_limit": n_tl,
        "pct_time_limit": p_tl,
        "mean_finished_sec": mean_finished,
        "std_finished_sec": std_finished,
        "mean_censored_sec": mean_cens,
        "std_censored_sec": std_cens
    }])

# ---------------------------------------
# Gráficos de tiempo y L1 + tiempo por n
# ---------------------------------------
def plot_runtime(df):
    import matplotlib.pyplot as plt

    # Histograma de tiempos
    if not df.empty:
        plt.figure()
        plt.hist(df["runtime_sec"], bins=20, color='skyblue', edgecolor='black')
        plt.xlabel("Tiempo (s)")
        plt.ylabel("Frecuencia")
        plt.title("Distribución de tiempos de ejecución")
        plt.grid(True)
        plt.show()
    else:
        print("DataFrame vacío, no se puede graficar el histograma de tiempos.")

    # Histograma de distancias L1
    # Convertir a numérico y eliminar NaNs para asegurar compatibilidad con np.isfinite
    dist = pd.to_numeric(df["dist_L1"], errors='coerce').dropna()

    if len(dist) > 0:
        plt.figure()
        plt.hist(dist, bins=20, edgecolor='black')
        plt.xlabel("Distancia L1")
        plt.ylabel("Frecuencia")
        plt.title("Distribución de dist_L1 (valores finitos)")
        plt.grid(True)
        plt.show()
    else:
        print("No hay valores finitos de dist_L1 para graficar.")

    # Promedio y desviación por número de variables n
    if not df.empty:
        grouped = df.groupby("n")["runtime_sec"].agg(["mean", "std"]).reset_index()

        plt.figure()
        plt.errorbar(
            grouped["n"], grouped["mean"], yerr=grouped["std"],
            fmt='o-', ecolor='red', capsize=5, markersize=6, color='blue'
        )
        plt.xlabel("Número de variables n")
        plt.ylabel("Tiempo promedio (s)")
        plt.yscale("log")  # <-- eje y en escala logarítmica
        plt.title("Tiempo promedio de ejecución \u00b1 desviación estándar")
        plt.grid(True)
        plt.show()
    else:
        print("DataFrame vacío, no se puede graficar el tiempo promedio por n.")

# Example call for the enumeration-based benchmark
# Be cautious with 'n' and 'bounds' as it can be very slow.
ns_enum = [6] # Keep 'n' small for enumeration
df_enum = benchmark_trial_knapsack_b_enumeration(ns_enum, n_zero = 1, trials_per_n=5)

print(df_enum.groupby(["n", "status", "verified_ok"]).size())
print(df_enum.describe())
summary_enum = summarize(df_enum)
print(summary_enum)
df_enum_opt = df_enum[df_enum["status"] == "OPTIMAL_WCE"].reset_index(drop=True)
plot_runtime(df_enum_opt)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def benchmark_grid_knapsack_b(
    ns,
    trials_per_setting=5,
    n_zero_values=(1, 2, 3, 4, 5, 6, 7, 8),
    base_seed=200,
):
    """
    Ejecuta benchmark y grafica:
    - Heatmap: tiempo promedio
    - Texto en cada celda: porcentaje de factibilidad
    """

    results = []
    seed = base_seed

    # =========================
    # 1. Ejecutar benchmarks
    # =========================
    for n in ns:
        for n_zero in n_zero_values:
            print(f"Ejecutando n={n}, n_zero={n_zero}")

            df = benchmark_trial_knapsack_b_enumeration(
                ns=[n],
                n_zero=n_zero,
                trials_per_n=trials_per_setting,
                base_seed=seed,
            )

            seed += trials_per_setting

            avg_time = df["runtime_sec"].mean()
            feasible_pct = 100.0 * (df["status"] == "OPTIMAL_WCE").mean()

            results.append({
                "n": n,
                "n_zero": n_zero,
                "avg_time_sec": avg_time,
                "feasible_pct": feasible_pct,
            })

    summary_df = pd.DataFrame(results)

    # =========================
    # 2. Construir matrices
    # =========================
    time_matrix = summary_df.pivot(
        index="n", columns="n_zero", values="avg_time_sec"
    )

    feasible_matrix = summary_df.pivot(
        index="n", columns="n_zero", values="feasible_pct"
    )

    # =========================
    # 3. Plot conjunto
    # =========================
    fig, ax = plt.subplots(figsize=(9, 5))

    im = ax.imshow(time_matrix.values)

    ax.set_xticks(range(len(time_matrix.columns)))
    ax.set_xticklabels(time_matrix.columns)
    ax.set_yticks(range(len(time_matrix.index)))
    ax.set_yticklabels(time_matrix.index)

    ax.set_xlabel("Número de variables forzadas a 0 (n_zero)")
    ax.set_ylabel("Número de variables (n)")
    ax.set_title("Tiempo promedio (color) y porcentaje factible (texto)")

    # Texto: tiempo + porcentaje
    for i in range(time_matrix.shape[0]):
        for j in range(time_matrix.shape[1]):
            t = time_matrix.iloc[i, j]
            p = feasible_matrix.iloc[i, j]

            ax.text(
                j, i,
                f"{t:.2f}s\n{p:.0f}%",
                ha="center",
                va="center",
                fontsize=10,
                color="white" if im.norm(t) > 0.5 else "black",
            )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Tiempo promedio (segundos)")

    plt.tight_layout()
    plt.show()

    return {
        "raw_summary": summary_df,
        "time_matrix": time_matrix,
        "feasible_matrix": feasible_matrix,
    }
ns = [14, 15, 16, 17 , 18 , 19, 20, 21, 22]
res = benchmark_grid_knapsack_b(
    ns=ns,
    trials_per_setting=30,
)

print("\nTiempo promedio (segundos):")
print(res["time_matrix"])

print("\nPorcentaje de instancias factibles (%):")
print(res["feasible_matrix"])


## Variar solo el vector a de restricciones

In [ ]:
def  generate_knapsack_a_only(
    n=10,
    n_zero=2,
    k_movable=1,
    rng=None,
):
    """
    Knapsack (a-only) CONSISTENTE con índices movibles/fijos
    - a^T x >= b (b fijo)
    - Solo k_movable coeficientes de a pueden variar
    - Los coeficientes movibles SOLO se eligen en variables que NO están en D
    - b se construye alcanzable con esos grados de libertad
    """

    if rng is None:
        rng = np.random.default_rng()

    assert n_zero < n, "n_zero debe ser < n"
    assert k_movable <= n - n_zero, \
        "k_movable no puede exceder variables NO anuladas por D"

    # =========================
    # 1. Dominio
    # =========================
    bounds = [(0, 10)] * n
    var_types = ["I"] * n

    # =========================
    # 2. Costos
    # =========================
    c_hat = rng.integers(1, 26, size=n)

    # =========================
    # 3. a nominal
    # =========================
    a_hat = rng.integers(1, 11, size=n)

    # =========================
    # 4. Conjunto favorable D
    # =========================
    D_mat = np.zeros((n_zero, n), dtype=int)
    d_vec = np.zeros(n_zero, dtype=int)

    zero_idx = np.arange(n_zero)          # x_i = 0
    nonzero_idx = np.arange(n_zero, n)    # x_i libres

    for i in range(n_zero):
        D_mat[i, i] = 1

    # =========================
    # 5. Elegir variables movibles SOLO en nonzero_idx
    # =========================
    movable_idx = rng.choice(nonzero_idx, size=k_movable, replace=False)
    movable_mask = np.zeros(n, dtype=bool)
    movable_mask[movable_idx] = True
    fixed_idx = np.where(~movable_mask)[0]

    # =========================
    # 6. Caja de a
    # =========================
    a_lb = a_hat.copy()
    a_ub = a_hat.copy()

    for i in movable_idx:
        a_lb[i] = 1
        a_ub[i] = 10

    # =========================
    # 7. Construir x_feasible ∈ D
    #    (usa SOLO variables no anuladas)
    # =========================
    x_feasible = np.zeros(n, dtype=int)
    for j in nonzero_idx:
        x_feasible[j] = rng.integers(3, 7)

    # =========================
    # 8. Construir b_hat ALCANZABLE
    # =========================
    lhs_lb = int(a_lb @ x_feasible)
    lhs_ub = int(a_ub @ x_feasible)

    # margen pequeño para evitar trivialidad
    if lhs_lb + 1 >= lhs_ub:
        b_hat = lhs_lb
    else:
        b_hat = rng.integers(lhs_lb + 1, lhs_ub)

    b_lb = b_hat
    b_ub = b_hat

    # =========================
    # 9. Compatibilidad X (dummy)
    # =========================
    A = np.ones((1, n), dtype=int)
    b_vec = np.array([0])

    return {
        "A": A,
        "b": b_vec,
        "c_hat": c_hat,
        "bounds": bounds,
        "var_types": var_types,
        "D_mat": D_mat,
        "d_vec": d_vec,

        "a_hat": a_hat,
        "a_lb": a_lb,
        "a_ub": a_ub,
        "b_hat": b_hat,
        "b_lb": b_lb,
        "b_ub": b_ub,

        # movilidad explícita
        "movable_idx": movable_idx,
        "fixed_idx": fixed_idx,
        "movable_mask": movable_mask,

        # debug
        "x_feasible": x_feasible,
        "nonzero_idx": nonzero_idx,
    }


In [ ]:
def solve_problem_8_a_only(
    v,
    c_hat,
    a_hat,
    b_hat,
    A,
    b_vec,
    bounds,
    var_types,
    d_star,
    D_mat,
    d_vec,
    a_lb,
    a_ub,
    movable_idx,
    fixed_idx,
    tolerance=1e-6,
):
    n = len(c_hat)
    Y = []

    while True:
        master = gp.Model("master_a_only")
        master.Params.OutputFlag = 0

        # =========================
        # x variables
        # =========================
        x = []
        for i in range(n):
            x.append(
                master.addVar(
                    lb=bounds[i][0],
                    ub=bounds[i][1],
                    vtype=GRB.INTEGER,
                )
            )

        # =========================
        # a variables (SOLO móviles)
        # =========================
        a = {}
        u = {}

        for i in movable_idx:
            a[i] = master.addVar(lb=a_lb[i], ub=a_ub[i], vtype=GRB.INTEGER)
            u[i] = master.addVar(lb=0)

            master.addConstr(a[i] - a_hat[i] <= u[i])
            master.addConstr(a_hat[i] - a[i] <= u[i])

        master.setObjective(
            gp.quicksum(u[i] for i in movable_idx), GRB.MINIMIZE
        )

        # =========================
        # Cortes acumulados
        # =========================
        for y in Y:
            master.addConstr(
                gp.quicksum(
                    (a[i] if i in movable_idx else a_hat[i]) * y[i]
                    for i in range(n)
                )
                <= b_hat - 1
            )

        # =========================
        # Restricciones
        # =========================
        master.addConstr(gp.quicksum(c_hat[i] * x[i] for i in range(n)) == v)

        master.addConstr(
            gp.quicksum(
                (a[i] if i in movable_idx else a_hat[i]) * x[i]
                for i in range(n)
            )
            >= b_hat
        )

        for i in range(A.shape[0]):
            master.addConstr(
                gp.quicksum(A[i, j] * x[j] for j in range(n)) >= b_vec[i]
            )

        for i in range(D_mat.shape[0]):
            master.addConstr(
                gp.quicksum(D_mat[i, j] * x[j] for j in range(n)) <= d_vec[i]
            )

        if np.isfinite(d_star):
            master.addConstr(
                gp.quicksum(u[i] for i in movable_idx) <= d_star
            )

        master.optimize()
        if master.status != GRB.OPTIMAL:
            return None

        a_sol = a_hat.copy()
        for i in movable_idx:
            a_sol[i] = int(a[i].X)

        # =========================
        # Separation
        # =========================
        sep = gp.Model()
        sep.Params.OutputFlag = 0

        y = [
            sep.addVar(lb=bounds[i][0], ub=bounds[i][1], vtype=GRB.INTEGER)
            for i in range(n)
        ]

        for i in range(A.shape[0]):
            sep.addConstr(
                gp.quicksum(A[i, j] * y[j] for j in range(n)) >= b_vec[i]
            )

        sep.addConstr(gp.quicksum(c_hat[i] * y[i] for i in range(n)) <= v - 1)

        sep.setObjective(
            gp.quicksum(a_sol[i] * y[i] for i in range(n)), GRB.MAXIMIZE
        )
        sep.optimize()

        if sep.status == GRB.OPTIMAL:
            y_sol = np.array([y[i].X for i in range(n)])
            if np.dot(a_sol, y_sol) >= b_hat - tolerance:
                Y.append(y_sol)
                continue

        return a_sol



In [ ]:
def compute_lower_bound_lemma4_L1_a_only(
    c_hat,
    v_bar,
    A,
    b_vec,
    bounds,
    var_types,
    a_hat,
    b_hat,
    a_lb,
    a_ub,
    time_limit=10,
    max_cuts=200,
):
    """
    Lemma 4 para el caso a-only (b fijo).
    """

    t0 = time.perf_counter()
    n = len(c_hat)

    m = gp.Model()
    m.Params.OutputFlag = 0
    m.Params.TimeLimit = time_limit

    a = m.addVars(n, lb=a_lb, ub=a_ub, vtype=GRB.INTEGER)
    u = m.addVars(n, lb=0)

    for i in range(n):
        m.addConstr(a[i] - a_hat[i] <= u[i])
        m.addConstr(a_hat[i] - a[i] <= u[i])

    m.setObjective(gp.quicksum(u[i] for i in range(n)), GRB.MINIMIZE)

    cuts = 0
    while cuts < max_cuts:
        if time.perf_counter() - t0 > time_limit:
            return np.inf

        m.optimize()
        if m.status != GRB.OPTIMAL:
            return np.inf

        a_val = np.array([a[i].X for i in range(n)])

        # Separation: buscar y mejor estricto
        sep = gp.Model()
        sep.Params.OutputFlag = 0
        sep.Params.TimeLimit = time_limit

        y = []
        for i in range(n):
            if var_types[i] == "B":
                y.append(sep.addVar(vtype=GRB.BINARY))
            elif var_types[i] == "I":
                y.append(
                    sep.addVar(lb=bounds[i][0], ub=bounds[i][1], vtype=GRB.INTEGER)
                )
            else:
                y.append(sep.addVar(lb=bounds[i][0], ub=bounds[i][1]))

        for i in range(A.shape[0]):
            sep.addConstr(gp.quicksum(A[i, j] * y[j] for j in range(n)) >= b_vec[i])

        sep.addConstr(gp.quicksum(c_hat[i] * y[i] for i in range(n)) <= v_bar - 1)

        sep.setObjective(gp.quicksum(a_val[i] * y[i] for i in range(n)), GRB.MAXIMIZE)
        sep.optimize()

        if sep.status != GRB.OPTIMAL or sep.ObjVal <= b_hat - 1 + 1e-6:
            break

        y_star = np.array([y[i].X for i in range(n)])
        m.addConstr(gp.quicksum(a[i] * y_star[i] for i in range(n)) <= b_hat - 1)
        cuts += 1

    return m.ObjVal

def compute_c_bounds_a_only(
    c_hat,
    A,
    b_vec,
    D_mat,
    d_vec,
    bounds,
    var_types,
    time_limit=5,
):
    """
    Cotas válidas de v = c^T x para el caso a-only.
    """

    n = len(c_hat)

    def build_model(sense):
        m = gp.Model()
        m.Params.OutputFlag = 0
        m.Params.TimeLimit = time_limit

        x = []
        for i in range(n):
            if var_types[i] == "B":
                x.append(m.addVar(vtype=GRB.BINARY))
            elif var_types[i] == "I":
                x.append(
                    m.addVar(lb=bounds[i][0], ub=bounds[i][1], vtype=GRB.INTEGER)
                )
            else:
                x.append(m.addVar(lb=bounds[i][0], ub=bounds[i][1]))

        # Restricción Ax ≥ b (problema observado)
        for i in range(A.shape[0]):
            m.addConstr(gp.quicksum(A[i, j] * x[j] for j in range(n)) >= b_vec[i])

        # Restricciones D
        for i in range(D_mat.shape[0]):
            m.addConstr(
                gp.quicksum(D_mat[i, j] * x[j] for j in range(n)) <= d_vec[i]
            )

        m.setObjective(gp.quicksum(c_hat[i] * x[i] for i in range(n)), sense)
        return m

    m_min = build_model(GRB.MINIMIZE)
    m_min.optimize()
    if m_min.status != GRB.OPTIMAL:
        raise RuntimeError("compute_c_bounds_a_only: problema base infactible")

    m_max = build_model(GRB.MAXIMIZE)
    m_max.optimize()
    if m_max.status != GRB.OPTIMAL:
        raise RuntimeError("compute_c_bounds_a_only: problema base infactible")

    return int(np.floor(m_min.ObjVal)), int(np.ceil(m_max.ObjVal))


def algorithm3_a_only(
    c_hat,
    a_hat,
    b_hat,
    A,
    b_vec,
    a_lb,
    a_ub,
    bounds,
    var_types,
    D_mat,
    d_vec,
    movable_idx,
    fixed_idx,
    time_limit_seconds=120,
):
    t0 = time.perf_counter()

    a_star_best = None
    d_star = np.inf
    stopped_by_lemma4 = False

    c_min, c_max = compute_c_bounds_a_only(
        c_hat, A, b_vec, D_mat, d_vec, bounds, var_types
    )

    for v in range(c_min, c_max + 1):
        if time.perf_counter() - t0 > time_limit_seconds:
            break

        a_sol = solve_problem_8_a_only(
            v,
            c_hat,
            a_hat,
            b_hat,
            A,
            b_vec,
            bounds,
            var_types,
            d_star,
            D_mat,
            d_vec,
            a_lb,
            a_ub,
            movable_idx,
            fixed_idx,
        )

        if a_sol is None:
            continue

        dist = np.sum(np.abs(a_sol[movable_idx] - a_hat[movable_idx]))

        if dist < d_star:
            d_star = dist
            a_star_best = a_sol

    return {
        "a_star_best": a_star_best,
        "b_star_best": b_hat,
        "d_star": d_star,
        "c_min": c_min,
        "c_max": c_max,
    }


In [ ]:
def run_trial_knapsack_a(n, n_zero, k_movable, seed = None, alg_time_limit=120):
    rng = np.random.default_rng()

    # =========================
    # 1. Generar instancia
    # =========================
    try:
        inst = generate_knapsack_a_only(n=n, n_zero=n_zero, k_movable= k_movable, rng=rng)
    except Exception as e:
        print ("Generation_ERROR")
        return {
            "seed": seed,
            "n": n,
            "runtime_sec": 0.0,
            "time_limit_sec": alg_time_limit,
            "status": "GENERATION_ERROR",
            "iters": 0,
            "d_star": None,
            "dist_L1": None,
            "verified_ok": False,
            "error": str(e)[:200],
        }

    # =========================
    # 2. Verificar X ∩ D ≠ ∅
    # =========================
    is_feas, _ = verify_instance_feasibility_alg3(inst)
    if not is_feas:
        print ("INSTANCE_INFEASIBLE")
        return {
            "seed": seed,
            "n": n,
            "runtime_sec": 0.0,
            "time_limit_sec": alg_time_limit,
            "status": "INSTANCE_INFEASIBLE",
            "iters": 0,
            "d_star": None,
            "dist_L1": None,
            "verified_ok": False,
            "error": "",
        }

    # =========================
    # 3. Ejecutar Algorithm 3-b
    # =========================
    t0 = time.perf_counter()
    try:
        res = algorithm3_a_only(
            c_hat=inst["c_hat"],
            a_hat=inst["a_hat"],
            b_hat=inst["b_hat"],
            A=inst["A"],
            b_vec=inst["b"],
            a_lb=inst["a_lb"],
            a_ub=inst["a_ub"],
            # b_lb=inst["b_lb"],
            # b_ub=inst["b_ub"],
            bounds=inst["bounds"],
            var_types=inst["var_types"],
            D_mat=inst["D_mat"],
            d_vec=inst["d_vec"],
            # sense="min",
            movable_idx = inst["movable_idx"],
            fixed_idx = inst["fixed_idx"],
            time_limit_seconds=alg_time_limit,
        )
    except Exception as e:
        runtime = time.perf_counter() - t0
        print ("NOT_SOLVED")
        return {
            "seed": seed,
            "n": n,
            "runtime_sec": runtime,
            "time_limit_sec": alg_time_limit,
            "status": "ALGORITHM_ERROR",
            "iters": 0,
            "d_star": None,
            "dist_L1": None,
            "verified_ok": False,
            "error": str(e)[:200],
        }

    runtime = time.perf_counter() - t0

    # =========================
    # 4. Interpretar resultado
    # =========================
    a_star = res.get("a_star_best")
    b_star = res.get("b_star_best")
    iters = res.get("iters", 0)
    stopped_by_lemma4 = res.get("stopped_by_lemma4", False)

    if a_star is not None:
        dist_L1 = np.sum(np.abs(a_star - inst["a_hat"]))
        algo_status = "OPTIMAL"
        print ("OPTIMAL")

    elif stopped_by_lemma4:
        dist_L1 = None
        algo_status = "NO_WCE_EXISTS"

    else:
        print ("WCE_INFEASIBLE")
        return {
            "seed": seed,
            "n": n,
            "runtime_sec": runtime,
            "time_limit_sec": alg_time_limit,
            "status": "INFEASIBLE",
            "iters": iters,
            "d_star": None,
            "dist_L1": None,
            "verified_ok": False,
            "error": "",
        }

    # =========================
    # 5. Verificación WCE
    # =========================
    try:
        ver = verify_algorithm3_solution(
            a_star=a_star,
            b_star=b_star,
            c_hat=inst["c_hat"],
            A=inst["A"],
            b_vec=inst["b"],
            D_mat=inst["D_mat"],
            d_vec=inst["d_vec"],
            bounds=inst["bounds"],
            var_types=inst["var_types"],
            tol=1e-6,
            verbose=False,
        )
        verified_ok = ver.get("verified_ok", False)
        verify_error = ver.get("reason", "")
    except Exception as e:
        # si la verificación falla, marcamos fallo de verificación
        verified_ok = False
        verify_error = str(e)[:200]

    final_status = (
        "OPTIMAL_WCE" if verified_ok else "OPTIMAL_NOT_WCE"
        if algo_status == "OPTIMAL"
        else algo_status
    )

    # =========================
    # 6. Salida final (consistente)
    # =========================
    return {
        "seed": seed,
        "n": n,
        "runtime_sec": runtime,
        "time_limit_sec": alg_time_limit,
        "status": final_status,
        "iters": iters,
        "d_star": None if res.get("d_star", np.inf) == np.inf else res.get("d_star"),
        "dist_L1": dist_L1,
        "verified_ok": verified_ok,
        "error": verify_error if not verified_ok else "",
    }


def benchmark_trial_knapsack_a(ns, n_zero, k_movable_val, trials_per_n=10, base_seed=200):
    """Igual que antes pero para Algorithm 3"""
    rows = []
    seed = base_seed

    total_trials = len(ns) * trials_per_n
    completed = 0

    for n in ns:
        for _ in range(trials_per_n):
            rows.append(run_trial_knapsack_a(n, n_zero, k_movable_val, seed))
            seed += 1
            completed += 1

            progress = completed / total_trials
            bar_length = 40
            filled_length = int(bar_length * progress)
            bar = "█" * filled_length + "-" * (bar_length - filled_length)
            print(f"\rProgreso: |{bar}| {progress*100:.1f}% ({completed}/{total_trials})", end="")

    print()
    return pd.DataFrame(rows)

In [ ]:
ns = [6]
df = benchmark_trial_knapsack_a (ns, n_zero = 2, k_movable_val = 1, trials_per_n=5)

print(df.groupby(["n", "status", "verified_ok"]).size())
print(df.describe())
summary = summarize(df)
print(summary)
df_opt = df[df["status"] == "OPTIMAL_WCE"].reset_index(drop=True)
plot_runtime(df_opt)

In [ ]:
def benchmark_grid_a_only(
    n,
    trials_per_cell=5,
    base_seed=200,
):
    """
    Meta-benchmark para el caso donde SOLO se mueve a.
    Usa benchmark_trial_knapsack_a internamente.
    """

    rows = []
    seed = base_seed

    for n_zero in range(1, 5):
        for k_movable_val in range(1, 5):

            print(f"\n▶ n_zero={n_zero}, k_movable_val={k_movable_val}")

            df = benchmark_trial_knapsack_a(
                ns=[n],
                n_zero=n_zero,
                k_movable_val=k_movable_val,
                trials_per_n=trials_per_cell,
                base_seed=seed,
            )

            seed += trials_per_cell

            df_ok = df[df["status"] == "OPTIMAL_WCE"]
            if len(df_ok) > 0:
                avg_runtime = df_ok["runtime_sec"].mean()
            else:
                avg_runtime = np.nan

            feasible_rate = np.mean(
                df["status"] == "OPTIMAL_WCE"
            )

            rows.append({
                "n": n,
                "n_zero": n_zero,
                "k_movable_val": k_movable_val,
                "avg_runtime": avg_runtime,
                "feasible_rate": feasible_rate,
            })

    return pd.DataFrame(rows)


In [ ]:
def build_confusion_matrices_a_only(df):
    """
    Filas    -> n_zero = 1..4
    Columnas -> k_movable_val = 1..4
    """

    mat_time = np.full((4, 4), np.nan)
    mat_feas = np.full((4, 4), np.nan)

    for _, row in df.iterrows():
        i = int(row["n_zero"]) - 1
        j = int(row["k_movable_val"]) - 1
        mat_time[i, j] = row["avg_runtime"]
        mat_feas[i, j] = row["feasible_rate"]

    return mat_time, mat_feas


In [ ]:
import matplotlib.pyplot as plt

def plot_confusion_matrix_a_only(mat_time, mat_feas, n):
    fig, ax = plt.subplots(figsize=(8, 6))

    im = ax.imshow(mat_time, origin="lower")

    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels([1, 2, 3, 4])
    ax.set_yticklabels([1, 2, 3, 4])

    ax.set_xlabel("Número de coeficientes de a movibles")
    ax.set_ylabel("n_zero")
    ax.set_title(f"Algorithm 3 – solo a movible (n = {n})")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Tiempo promedio (s)")

    for i in range(4):
        for j in range(4):
            if np.isnan(mat_time[i, j]):
                continue
            txt = (
                f"{mat_time[i, j]:.2f}s\n"
                f"{100 * mat_feas[i, j]:.0f}%"
            )
            ax.text(j, i, txt, ha="center", va="center", color="white", fontsize=10)

    plt.tight_layout()
    plt.show()


In [ ]:
ns = [8,9,10,11,12]
for n in ns:
    df_grid_a = benchmark_grid_a_only(
    n=n,
    trials_per_cell=20,)
    mat_time_a, mat_feas_a = build_confusion_matrices_a_only(df_grid_a)
    plot_confusion_matrix_a_only(mat_time_a, mat_feas_a, n)
